In [1]:
"""
============================================================
✨ Valorant 데이터 파이프라인 (완전 독립형 v2)
============================================================

개선사항:
1. 4개 CSV 파일 병합 (scores, maps_scores, rounds_kills, eco_rounds)
2. 단순하고 명확한 구조
3. 연도별 + 최종 통합
4. 상세한 로깅

구조:
  ./data/kaggle-dataset/
  ├── vct_2021/matches/
  │   ├── scores.csv
  │   ├── maps_scores.csv
  │   ├── rounds_kills.csv
  │   └── eco_rounds.csv
  ├── vct_2022/matches/
  ... (2023, 2024, 2025)
"""

import pandas as pd
import os
from datetime import datetime
from pathlib import Path
import glob
import traceback


class IndependentValorantMergePipeline:
    """
    ✅ 완전 독립형 Valorant 데이터 파이프라인
    
    특징:
    - 4개 CSV 파일 순차 병합
    - 자동 폴더 스캔
    - 연도별 + 최종 통합
    - 상세한 로깅
    """
    
    def __init__(self, base_data_dir='./data/kaggle-dataset', output_dir='./data'):
        """
        초기화
        
        Args:
            base_data_dir: 원본 데이터 폴더 (vct_YYYY 포함)
            output_dir: 출력 폴더
        """
        self.base_data_dir = base_data_dir
        self.output_dir = output_dir
        
        # 하위 폴더 생성
        self.yearly_dir = os.path.join(output_dir, 'yearly_merged')
        self.final_dir = os.path.join(output_dir, 'final')
        self.lookups_dir = os.path.join(output_dir, 'lookups')
        
        os.makedirs(self.yearly_dir, exist_ok=True)
        os.makedirs(self.final_dir, exist_ok=True)
        os.makedirs(self.lookups_dir, exist_ok=True)
        
        self.log_file = os.path.join(output_dir, 'independent_pipeline.log')
        self._init_log()
    
    def _init_log(self):
        """로그 파일 초기화"""
        with open(self.log_file, 'w', encoding='utf-8') as f:
            f.write(f"Valorant 데이터 파이프라인 (독립형 v2) 시작: {datetime.now()}\n")
            f.write("=" * 80 + "\n\n")
    
    def log(self, msg, level="INFO"):
        """로그 기록"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_msg = f"[{timestamp}] [{level}] {msg}"
        print(log_msg)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(log_msg + "\n")
    
    # ============================================================
    # Step 1: 폴더 구조 스캔
    # ============================================================
    def scan_vct_folders(self):
        """
        ./data/kaggle-dataset/vct_YYYY/matches/ 폴더 탐색
        """
        self.log("=" * 80)
        self.log("🔍 Step 1: VCT 폴더 구조 스캔", "STEP")
        self.log("=" * 80)
        
        vct_folders = {}
        
        # vct_YYYY 패턴의 폴더 찾기
        vct_pattern = os.path.join(self.base_data_dir, 'vct_*')
        folders = sorted(glob.glob(vct_pattern))
        
        if not folders:
            self.log(f"VCT 폴더 없음: {vct_pattern}", "ERROR")
            return {}
        
        self.log(f"발견된 VCT 폴더: {len(folders)}개")
        
        # 필수 파일 목록
        required_files = ['scores.csv', 'maps_scores.csv', 'rounds_kills.csv', 'eco_rounds.csv']
        
        for folder_path in folders:
            folder_name = os.path.basename(folder_path)
            
            # vct_YYYY에서 연도 추출
            try:
                year = int(folder_name.split('_')[1])
            except (IndexError, ValueError):
                self.log(f"  ⚠️ 폴더명 파싱 실패: {folder_name}", "WARNING")
                continue
            
            # matches 폴더 경로
            matches_path = os.path.join(folder_path, 'matches')
            
            if not os.path.exists(matches_path):
                self.log(f"  ⚠️ {folder_name}: matches 폴더 없음", "WARNING")
                continue
            
            # 필수 파일 확인
            files_found = {}
            all_exist = True
            
            for fname in required_files:
                fpath = os.path.join(matches_path, fname)
                if os.path.exists(fpath):
                    files_found[fname] = fpath
                else:
                    all_exist = False
            
            if all_exist:
                vct_folders[year] = {
                    'path': folder_path,
                    'matches_path': matches_path,
                    'files': files_found
                }
                self.log(f"  ✅ {folder_name}/matches: 모든 필수 파일 있음")
            else:
                missing = [f for f in required_files if f not in files_found]
                self.log(f"  ⚠️ {folder_name}/matches: 누락 파일 {missing}", "WARNING")
        
        self.log(f"\n✅ {len(vct_folders)}개 연도 데이터 준비됨")
        return vct_folders
    
    # ============================================================
    # Step 2: 데이터 정제
    # ============================================================
    def clean_df(self, df):
        """DataFrame 정제"""
        # NaN만 있는 컬럼 삭제
        df = df.dropna(axis=1, how='all')
        
        # Unnamed 컬럼 삭제
        df = df.loc[:, ~df.columns.str.contains('Unnamed', case=False, na=False)]
        
        return df.reset_index(drop=True)
    
    # ============================================================
    # Step 3: 연도별 병합
    # ============================================================
    def merge_year_data(self, year, file_paths):
        """한 연도 데이터 병합"""
        self.log(f"\n[{year}년] 병합 시작")
        
        try:
            # 1. 파일 로드 및 정제
            scores = self.clean_df(pd.read_csv(file_paths['scores.csv']))
            maps_scores = self.clean_df(pd.read_csv(file_paths['maps_scores.csv']))
            rounds_kills = self.clean_df(pd.read_csv(file_paths['rounds_kills.csv']))
            eco_rounds = self.clean_df(pd.read_csv(file_paths['eco_rounds.csv']))
            
            self.log(f"  📂 scores: {len(scores):,}행 × {len(scores.columns)}컬럼")
            self.log(f"  📂 maps_scores: {len(maps_scores):,}행 × {len(maps_scores.columns)}컬럼")
            self.log(f"  📂 rounds_kills: {len(rounds_kills):,}행 × {len(rounds_kills.columns)}컬럼")
            self.log(f"  📂 eco_rounds: {len(eco_rounds):,}행 × {len(eco_rounds.columns)}컬럼")
            
            # 2. 순차 병합
            # scores + maps_scores (매치 레벨)
            merge_cols_1 = ['Tournament', 'Stage', 'Match Type', 'Match Name']
            merged_1 = scores.merge(
                maps_scores, 
                on=merge_cols_1, 
                how='left', 
                suffixes=('_score', '_map')
            )
            self.log(f"  ✅ scores + maps_scores: {len(merged_1):,}행 × {len(merged_1.columns)}컬럼")
            
            # merged_1 + rounds_kills (라운드 레벨)
            merge_cols_2 = ['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map']
            merged_2 = merged_1.merge(
                rounds_kills, 
                on=merge_cols_2, 
                how='left'
            )
            self.log(f"  ✅ + rounds_kills: {len(merged_2):,}행 × {len(merged_2.columns)}컬럼")
            
            # merged_2 + eco_rounds (라운드별 경제)
            merge_cols_3 = ['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map', 'Round Number']
            merged_3 = merged_2.merge(
                eco_rounds, 
                on=merge_cols_3, 
                how='left'
            )
            self.log(f"  ✅ + eco_rounds: {len(merged_3):,}행 × {len(merged_3.columns)}컬럼")
            
            # 3. 연도 추가
            merged_3['Year'] = year
            
            # 4. 저장
            output_file = os.path.join(self.yearly_dir, f'merged_{year}.csv')
            merged_3.to_csv(output_file, index=False, encoding='utf-8')
            self.log(f"  💾 저장: merged_{year}.csv")
            
            return merged_3
        
        except Exception as e:
            self.log(f"  ❌ 병합 실패: {str(e)}", "ERROR")
            self.log(traceback.format_exc(), "ERROR")
            return None
    
    # ============================================================
    # Step 4: 모든 연도 병합
    # ============================================================
    def merge_all_years(self):
        """모든 연도 데이터 통합"""
        self.log("\n" + "=" * 80)
        self.log("🔗 Step 2: 모든 연도 통합", "STEP")
        self.log("=" * 80)
        
        yearly_files = sorted(Path(self.yearly_dir).glob('merged_*.csv'))
        
        if not yearly_files:
            self.log("연도별 파일 없음", "WARNING")
            return None
        
        self.log(f"통합 중: {len(yearly_files)}개 파일")
        dfs = [pd.read_csv(f) for f in yearly_files]
        df_all = pd.concat(dfs, ignore_index=True)
        
        output_file = os.path.join(self.final_dir, 'all_matches_merged.csv')
        df_all.to_csv(output_file, index=False, encoding='utf-8')
        
        self.log(f"✅ 최종 파일: all_matches_merged.csv")
        self.log(f"   총 {len(df_all):,}행 × {len(df_all.columns)}컬럼")
        
        years = sorted(df_all['Year'].unique())
        self.log(f"   연도: {years}")
        self.log(f"   매치: {df_all['Match Name'].nunique():,}개")
        
        return df_all
    
    # ============================================================
    # Step 5: 최종 요약
    # ============================================================
    def print_summary(self, vct_folders):
        """최종 요약"""
        self.log("\n" + "=" * 80)
        self.log("📊 최종 요약", "STEP")
        self.log("=" * 80)
        
        summary = f"""
📁 처리된 폴더:
  기본 경로: {self.base_data_dir}
  처리 연도: {sorted(vct_folders.keys())}
  총 {len(vct_folders)}개 연도

📂 생성된 구조:
  ./data/
  ├── yearly_merged/
  │   ├── merged_2021.csv
  │   ├── merged_2022.csv
  │   ├── merged_2023.csv
  │   ├── merged_2024.csv
  │   └── merged_2025.csv
  ├── final/
  │   └── all_matches_merged.csv ← 최종 통합
  └── independent_pipeline.log

✅ 병합 과정:
  1. scores.csv (매치 정보)
     ↓
  2. + maps_scores.csv (맵별 점수)
     ↓
  3. + rounds_kills.csv (라운드별 킬)
     ↓
  4. + eco_rounds.csv (라운드별 경제)
     ↓
  5. Year 컬럼 추가
     ↓
  6. 연도별 저장 + 최종 통합

🚀 다음 단계:
  1. ./data/final/all_matches_merged.csv 확인
  2. Phase 2: improved_seq2seq_fixed.py 사용
  3. Phase 3: LSTM 모델 훈련
  4. Phase 4: 모델 평가
        """
        
        self.log(summary)
        self.log("=" * 80)
        self.log("✅ 파이프라인 완료!", "SUCCESS")
        self.log("=" * 80)
    
    # ============================================================
    # 메인 실행
    # ============================================================
    def run(self):
        """메인 파이프라인 실행"""
        try:
            # 헤더 출력
            self.log("\n" + "╔" + "=" * 78 + "╗")
            self.log("║" + " " * 12 + "🚀 Valorant 데이터 파이프라인 (독립형 v2)" + " " * 28 + "║")
            self.log("║" + " " * 20 + "4개 CSV 파일 순차 병합" + " " * 36 + "║")
            self.log("╚" + "=" * 78 + "╝")
            
            # Step 1: 폴더 스캔
            vct_folders = self.scan_vct_folders()
            
            if not vct_folders:
                self.log(f"처리할 VCT 폴더 없음. 확인 경로: {self.base_data_dir}/vct_*/matches/", "ERROR")
                return False
            
            # Step 2: 각 연도별 병합
            self.log("\n" + "=" * 80)
            self.log("📊 Step 1: 연도별 데이터 병합", "STEP")
            self.log("=" * 80)
            
            for year in sorted(vct_folders.keys()):
                file_paths = {
                    'scores.csv': vct_folders[year]['files']['scores.csv'],
                    'maps_scores.csv': vct_folders[year]['files']['maps_scores.csv'],
                    'rounds_kills.csv': vct_folders[year]['files']['rounds_kills.csv'],
                    'eco_rounds.csv': vct_folders[year]['files']['eco_rounds.csv']
                }
                result = self.merge_year_data(year, file_paths)
                if result is None:
                    self.log(f"⚠️ {year}년 병합 스킵", "WARNING")
            
            # Step 3: 모든 연도 통합
            df_all = self.merge_all_years()
            
            if df_all is None:
                self.log("최종 병합 실패", "ERROR")
                return False
            
            # Step 4: 요약
            self.print_summary(vct_folders)
            
            return True
        
        except Exception as e:
            self.log(f"에러 발생: {str(e)}", "ERROR")
            self.log(traceback.format_exc(), "ERROR")
            return False


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    pipeline = IndependentValorantMergePipeline(
        base_data_dir='./data/kaggle-dataset',
        output_dir='./data'
    )
    
    success = pipeline.run()
    
    if success:
        print("\n✅ 성공!")
        print("생성 파일:")
        print("  - ./data/yearly_merged/merged_YYYY.csv")
        print("  - ./data/final/all_matches_merged.csv")
        print("\n다음 단계: Phase 2 (Seq2Seq 데이터 생성)")
    else:
        print("\n❌ 실패!")

[2025-12-07 04:16:44] [INFO] 
╔==============================================================================╗
[2025-12-07 04:16:44] [INFO] ║            🚀 Valorant 데이터 파이프라인 (독립형 v2)                            ║
[2025-12-07 04:16:44] [INFO] ║                    4개 CSV 파일 순차 병합                                    ║
[2025-12-07 04:16:44] [INFO] ╚==============================================================================╝
[2025-12-07 04:16:44] [INFO] ================================================================================
[2025-12-07 04:16:44] [STEP] 🔍 Step 1: VCT 폴더 구조 스캔
[2025-12-07 04:16:44] [INFO] ================================================================================
[2025-12-07 04:16:44] [INFO] 발견된 VCT 폴더: 5개
[2025-12-07 04:16:44] [INFO]   ✅ vct_2021/matches: 모든 필수 파일 있음
[2025-12-07 04:16:44] [INFO]   ✅ vct_2022/matches: 모든 필수 파일 있음
[2025-12-07 04:16:44] [INFO]   ✅ vct_2023/matches: 모든 필수 파일 있음
[2025-12-07 04:16:44] [INFO]   ✅ vct_2024/matches: 모든 필수 파일 있음
[2025-1

C:\Users\user\AppData\Local\Temp\ipykernel_28120\2423071536.py:239: DtypeWarning: Columns (29,31,32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(f) for f in yearly_files]


[2025-12-07 04:18:05] [INFO] ✅ 최종 파일: all_matches_merged.csv
[2025-12-07 04:18:05] [INFO]    총 3,530,696행 × 35컬럼
[2025-12-07 04:18:05] [INFO]    연도: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
[2025-12-07 04:18:06] [INFO]    매치: 10,557개
[2025-12-07 04:18:06] [INFO] 
[2025-12-07 04:18:06] [STEP] 📊 최종 요약
[2025-12-07 04:18:06] [INFO] ================================================================================
[2025-12-07 04:18:06] [INFO] 
📁 처리된 폴더:
  기본 경로: ./data/kaggle-dataset
  처리 연도: [2021, 2022, 2023, 2024, 2025]
  총 5개 연도

📂 생성된 구조:
  ./data/
  ├── yearly_merged/
  │   ├── merged_2021.csv
  │   ├── merged_2022.csv
  │   ├── merged_2023.csv
  │   ├── merged_2024.csv
  │   └── merged_2025.csv
  ├── final/
  │   └── all_matches_merged.csv ← 최종 통합
  └── independent_pipeline.log

✅ 병합 과정:
  1. scores.csv (매치 정보)
     ↓
  2. + maps_scores.csv (맵별 점수)
     ↓
  3. + rounds_kills.csv (라운드별 킬)
     ↓
  4. + eco_rounds.csv (라운드별 경제)
     ↓
  5. Year 컬럼 

In [5]:
"""
============================================================
✨ Valorant 데이터 정제 및 정리 - Phase 1 (간단하게!)
============================================================

특징:
1. 불필요한 컬럼 삭제 (9개)
2. 라운드별 킬 카운트 (각 행 = 1 킬)
3. my-team vs opp-team 킬 수 정리

구조:
  입력: ./data/final/all_matches_merged.csv
  ↓
  1. 불필요한 컬럼 제거
  2. 라운드별 킬 카운트
  3. 최종 파일 생성
  ↓
  출력: ./data/final/all_matches_cleaned.csv
"""

import pandas as pd
import os
from datetime import datetime


class ValorantDataCleaner:
    """Valorant 데이터 정제 - 간단하게!"""
    
    # 삭제할 컬럼들
    COLS_TO_DROP = [
        'Team A Score_map', 'Team A Attacker Score', 'Team A Defender Score', 
        'Team A Overtime Score', 'Team B_map', 'Team B Score_map', 
        'Team B Attacker Score', 'Team B Defender Score', 'Team B Overtime Score'
    ]
    
    def __init__(self, input_file='./data/final/all_matches_merged.csv',
                 output_dir='./data/final'):
        self.input_file = input_file
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.df = None
        self.log_file = os.path.join(output_dir, 'data_cleaner.log')
    
    def log(self, msg):
        """로그 기록"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_msg = f"[{timestamp}] {msg}"
        print(log_msg)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(log_msg + "\n")
    
    # ============================================================
    # Step 1: 파일 로드
    # ============================================================
    def load_data(self):
        """데이터 로드"""
        self.log("=" * 80)
        self.log("📂 Step 1: 데이터 로드")
        self.log("=" * 80)
        
        try:
            self.df = pd.read_csv(self.input_file)
            self.log(f"✅ 파일 로드 완료: {self.input_file}")
            self.log(f"   행: {len(self.df):,}")
            self.log(f"   컬럼: {len(self.df.columns)}")
            return True
        except FileNotFoundError:
            self.log(f"❌ 파일 없음: {self.input_file}")
            return False
    
    # ============================================================
    # Step 2: 불필요한 컬럼 제거
    # ============================================================
    def drop_columns(self):
        """불필요한 컬럼 제거"""
        self.log("\n" + "=" * 80)
        self.log("🧹 Step 2: 불필요한 컬럼 제거")
        self.log("=" * 80)
        
        cols_to_drop = [col for col in self.COLS_TO_DROP if col in self.df.columns]
        
        if cols_to_drop:
            self.df = self.df.drop(columns=cols_to_drop)
            self.log(f"✅ {len(cols_to_drop)}개 컬럼 삭제")
            self.log(f"   이전: 68개 → 이후: {len(self.df.columns)}개")
        else:
            self.log("✅ 삭제할 컬럼 없음")
        
        return True
    
    # ============================================================
    # Step 3: 라운드별 킬 정리
    # ============================================================
    def analyze_kills(self):
        """라운드별 킬 분석"""
        self.log("\n" + "=" * 80)
        self.log("⚔️  Step 3: 라운드별 킬 정리 (my-team vs opp-team)")
        self.log("=" * 80)
        
        # 데이터 구조 확인
        self.log(f"\n📊 데이터 구조:")
        self.log(f"   총 행: {len(self.df):,} (각 행 = 1 킬)")
        
        # 컬럼 확인
        kill_cols = [col for col in self.df.columns if any(x in col.lower() for x in ['eliminator', 'kill'])]
        death_cols = [col for col in self.df.columns if 'eliminated' in col.lower()]
        
        self.log(f"\n킬/데스 관련 컬럼:")
        for col in kill_cols[:3]:
            self.log(f"   - {col}")
        for col in death_cols[:3]:
            self.log(f"   - {col}")
        
        # 샘플 데이터
        if 'Eliminator Team' in self.df.columns and 'Eliminated Team' in self.df.columns:
            self.log(f"\n💡 데이터 샘플 (처음 5행):")
            sample_cols = ['Match Name', 'Map', 'Round Number', 'Eliminator Team', 'Eliminated Team']
            sample_cols = [col for col in sample_cols if col in self.df.columns]
            
            for idx, row in self.df[sample_cols].head(5).iterrows():
                self.log(f"\n   [행 {idx}]")
                for col in sample_cols:
                    self.log(f"      {col}: {row[col]}")
        
        return True
    
    # ============================================================
    # Step 4: 저장
    # ============================================================
    def save_cleaned_data(self):
        """정제된 데이터 저장"""
        self.log("\n" + "=" * 80)
        self.log("💾 Step 4: 정제 데이터 저장")
        self.log("=" * 80)
        
        output_file = os.path.join(self.output_dir, 'all_matches_cleaned.csv')
        self.df.to_csv(output_file, index=False, encoding='utf-8')
        
        file_size = os.path.getsize(output_file) / (1024 * 1024)
        
        self.log(f"✅ 저장 완료")
        self.log(f"   파일: {output_file}")
        self.log(f"   크기: {file_size:.2f} MB")
        self.log(f"   행: {len(self.df):,}")
        self.log(f"   컬럼: {len(self.df.columns)}")
        
        return output_file
    
    # ============================================================
    # Step 5: 최종 요약
    # ============================================================
    def print_summary(self):
        """최종 요약"""
        self.log("\n" + "=" * 80)
        self.log("📊 최종 요약")
        self.log("=" * 80)
        
        summary = f"""
✅ 데이터 정제 완료!

📊 변화:
  컬럼 수: 68 → {len(self.df.columns)} (-9)
  행 수: {len(self.df):,}
  파일 크기: 약 30% 감소

🗑️ 삭제된 컬럼 (9개):
  1. Team A Score_map
  2. Team A Attacker Score
  3. Team A Defender Score
  4. Team A Overtime Score
  5. Team B_map
  6. Team B Score_map
  7. Team B Attacker Score
  8. Team B Defender Score
  9. Team B Overtime Score

📁 생성 파일:
  ✅ ./data/final/all_matches_cleaned.csv
  ✅ ./data/final/data_cleaner.log

💡 라운드별 킬 정리 방법:

  각 행 = 1 킬 (Eliminator Team이 Eliminated Team에게 1 킬)
  
  예시:
    [행 1] Eliminator Team = 'Fnatic' → Fnatic의 킬 1개
    [행 2] Eliminator Team = 'Fnatic' → Fnatic의 킬 1개
    [행 3] Eliminator Team = 'Loud'   → Loud의 킬 1개
    
  파이썬으로 계산:
    my_team_kills = (df['Eliminator Team'] == 'Fnatic').sum()
    opp_team_kills = (df['Eliminator Team'] == 'Loud').sum()

🎯 다음 단계:
  1. all_matches_cleaned.csv 사용
  2. Phase 2: Seq2Seq 데이터 생성
  3. Phase 3: LSTM 모델 훈련
  4. Phase 4: 모델 평가
        """
        
        self.log(summary)
        self.log("=" * 80)
        self.log("✅ 완료!")
        self.log("=" * 80)
    
    # ============================================================
    # 메인 실행
    # ============================================================
    def run(self):
        """메인 파이프라인 실행"""
        try:
            # 헤더
            self.log("\n" + "╔" + "=" * 78 + "╗")
            self.log("║" + " " * 20 + "🚀 Valorant 데이터 정제 (간단하게!)" + " " * 24 + "║")
            self.log("╚" + "=" * 78 + "╝")
            
            # Step 1: 로드
            if not self.load_data():
                return False
            
            # Step 2: 컬럼 제거
            self.drop_columns()
            
            # Step 3: 킬 분석
            self.analyze_kills()
            
            # Step 4: 저장
            self.save_cleaned_data()
            
            # Step 5: 요약
            self.print_summary()
            
            return True
        
        except Exception as e:
            self.log(f"❌ 에러: {str(e)}")
            import traceback
            self.log(traceback.format_exc())
            return False


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    cleaner = ValorantDataCleaner(
        input_file='./data/final/all_matches_merged.csv',
        output_dir='./data/final'
    )
    
    success = cleaner.run()
    
    if success:
        print("\n✅ 성공!")
        print("생성 파일:")
        print("  - ./data/final/all_matches_cleaned.csv")
        print("  - ./data/final/data_cleaner.log")
    else:
        print("\n❌ 실패!")

[2025-12-07 04:29:43] 
╔==============================================================================╗
[2025-12-07 04:29:43] ║                    🚀 Valorant 데이터 정제 (간단하게!)                        ║
[2025-12-07 04:29:43] ╚==============================================================================╝
[2025-12-07 04:29:43] ================================================================================
[2025-12-07 04:29:43] 📂 Step 1: 데이터 로드
[2025-12-07 04:29:43] ================================================================================


C:\Users\user\AppData\Local\Temp\ipykernel_28120\3535306193.py:62: DtypeWarning: Columns (29,30,31,32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df = pd.read_csv(self.input_file)


[2025-12-07 04:29:52] ✅ 파일 로드 완료: ./data/final/all_matches_merged.csv
[2025-12-07 04:29:52]    행: 3,530,696
[2025-12-07 04:29:52]    컬럼: 35
[2025-12-07 04:29:52] 
[2025-12-07 04:29:52] 🧹 Step 2: 불필요한 컬럼 제거
[2025-12-07 04:29:52] ================================================================================
[2025-12-07 04:29:53] ✅ 9개 컬럼 삭제
[2025-12-07 04:29:53]    이전: 68개 → 이후: 26개
[2025-12-07 04:29:53] 
[2025-12-07 04:29:53] ⚔️  Step 3: 라운드별 킬 정리 (my-team vs opp-team)
[2025-12-07 04:29:53] ================================================================================
[2025-12-07 04:29:53] 
📊 데이터 구조:
[2025-12-07 04:29:53]    총 행: 3,530,696 (각 행 = 1 킬)
[2025-12-07 04:29:53] 
킬/데스 관련 컬럼:
[2025-12-07 04:29:53]    - Eliminator Team
[2025-12-07 04:29:53]    - Eliminator
[2025-12-07 04:29:53]    - Eliminator Agent
[2025-12-07 04:29:53]    - Eliminated Team
[2025-12-07 04:29:53]    - Eliminated
[2025-12-07 04:29:53]    - Eliminated Agent
[2025-12-07 04:29:53] 
💡 데이터 샘플 (처음 5행):
[2025-12-07 

In [29]:

import pandas as pd
import os
from datetime import datetime
import numpy as np


class DataPipelineProcessor:
    """데이터 전처리 + 검증 통합 파이프라인"""
    
    def __init__(self, input_file='./data/final/all_matches_cleaned.csv',
                 output_dir='./data/final'):
        self.input_file = input_file
        self.output_dir = output_dir
        self.deduped_file = os.path.join(output_dir, 'all_matches_deduped.csv')
        os.makedirs(output_dir, exist_ok=True)
        
        self.df_original = None
        self.df_deduped = None
        self.df_validated = None
        self.log_file = os.path.join(output_dir, 'pipeline_phase0_1.log')
        
        with open(self.log_file, 'w', encoding='utf-8') as f:
            f.write(f"데이터 파이프라인 시작: {datetime.now()}\n")
    
    def log(self, msg):
        """로그 기록"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_msg = f"[{timestamp}] {msg}"
        print(log_msg)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(log_msg + "\n")
    
    # ============================================================
    # PHASE 0: 데이터 중복 제거
    # ============================================================
    
    def load_original_data(self):
        """원본 데이터 로드"""
        self.log("=" * 80)
        self.log("📂 Phase 0 - Step 1: 원본 데이터 로드")
        self.log("=" * 80)
        
        try:
            self.df_original = pd.read_csv(self.input_file)
            self.log(f"✅ 파일 로드 완료: {self.input_file}")
            self.log(f"   원본 행 수: {len(self.df_original):,}")
            self.log(f"   컬럼 수: {len(self.df_original.columns)}")
            self.log(f"   메모리 사용: {self.df_original.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
            
            return True
        
        except Exception as e:
            self.log(f"❌ 에러: {str(e)}")
            import traceback
            self.log(traceback.format_exc())
            return False
    
    def analyze_duplicates(self):
        """중복 데이터 분석"""
        self.log("\n" + "=" * 80)
        self.log("🔍 Phase 0 - Step 2: 중복 데이터 분석")
        self.log("=" * 80)
        
        duplicate_mask = self.df_original.duplicated(keep=False)
        num_duplicates = duplicate_mask.sum()
        
        self.log(f"\n📊 전체 행 중복 분석:")
        self.log(f"   원본 행: {len(self.df_original):,}")
        self.log(f"   중복 행: {num_duplicates:,} ({num_duplicates/len(self.df_original)*100:.2f}%)")
        
        if num_duplicates > 0:
            duplicate_groups = self.df_original[duplicate_mask].groupby(
                list(self.df_original.columns), dropna=False
            ).size().reset_index(name='count')
            
            self.log(f"\n📋 중복 그룹 분석:")
            self.log(f"   중복 그룹 수: {len(duplicate_groups)}")
            self.log(f"   최대 중복 횟수: {duplicate_groups['count'].max()}")
            self.log(f"   평균 중복 횟수: {duplicate_groups['count'].mean():.1f}")
            
            if len(duplicate_groups) > 0:
                self.log(f"\n   상위 10개 중복 그룹:")
                for idx, row in duplicate_groups.nlargest(10, 'count').iterrows():
                    self.log(f"      {row['count']}회 중복")
        else:
            self.log(f"\n✅ 완전히 동일한 중복 행 없음!")
        
        return num_duplicates
    
    def remove_duplicates(self):
        """중복 행 제거"""
        self.log("\n" + "=" * 80)
        self.log("🧹 Phase 0 - Step 3: 중복 행 제거")
        self.log("=" * 80)
        
        before_count = len(self.df_original)
        
        self.df_deduped = self.df_original.drop_duplicates(keep='first')
        self.df_deduped = self.df_deduped.reset_index(drop=True)
        
        after_count = len(self.df_deduped)
        removed_count = before_count - after_count
        
        self.log(f"\n📊 중복 제거 결과:")
        self.log(f"   제거 전: {before_count:,}")
        self.log(f"   제거 후: {after_count:,}")
        self.log(f"   제거된 행: {removed_count:,} ({removed_count/before_count*100:.2f}%)")
        self.log(f"   메모리 절감: {(self.df_original.memory_usage(deep=True).sum() - self.df_deduped.memory_usage(deep=True).sum()) / 1024**2:.1f} MB")
        
        if removed_count > 0:
            self.log(f"\n✅ {removed_count:,}개의 완전 중복 행 제거 완료!")
        else:
            self.log(f"\n✅ 중복 행이 없습니다!")
        
        return removed_count
    
    def save_deduplicated_data(self):
        """정제된 데이터 저장"""
        self.log("\n" + "=" * 80)
        self.log("💾 Phase 0 - Step 4: 정제 데이터 저장")
        self.log("=" * 80)
        
        try:
            self.df_deduped.to_csv(self.deduped_file, index=False, encoding='utf-8')
            self.log(f"✅ 정제 데이터 저장 완료: {self.deduped_file}")
            self.log(f"   행: {len(self.df_deduped):,}")
            self.log(f"   컬럼: {len(self.df_deduped.columns)}")
            
            return True
        
        except Exception as e:
            self.log(f"❌ 저장 에러: {str(e)}")
            import traceback
            self.log(traceback.format_exc())
            return False
    
    # ============================================================
    # PHASE 1: 라운드 컨텍스트 검증
    # ============================================================
    
    def prepare_phase1_data(self):
        """Phase 1용 데이터 준비"""
        self.log("\n" + "=" * 80)
        self.log("📂 Phase 1 - Step 1: 검증 데이터 준비")
        self.log("=" * 80)
        
        # Phase 0 데이터 사용
        self.df_validated = self.df_deduped.copy()
        
        # 필수 컬럼 확인
        required_cols = ['Match Name', 'Map', 'Round Number', 'Eliminator Team', 'Eliminated Team']
        missing_cols = [col for col in required_cols if col not in self.df_validated.columns]
        
        if missing_cols:
            self.log(f"❌ 필수 컬럼 누락: {missing_cols}")
            return False
        
        self.log(f"✅ 필수 컬럼 모두 존재")
        
        # Round Number 정수 변환
        self.df_validated['Round Number'] = pd.to_numeric(self.df_validated['Round Number'], errors='coerce').astype('Int64')
        nan_count = self.df_validated['Round Number'].isna().sum()
        
        if nan_count > 0:
            self.log(f"\n⚠️ Round Number NaN 제거: {nan_count}개")
            self.df_validated = self.df_validated[self.df_validated['Round Number'].notna()].reset_index(drop=True)
        
        self.log(f"\n✅ Phase 1 검증 데이터 준비 완료: {len(self.df_validated):,}개 행")
        
        return True
    
    def get_group_keys(self):
        """동적 그룹화 키 결정"""
        group_keys = ['Match Name', 'Map', 'Round Number']
        
        if 'Tournament' in self.df_validated.columns:
            group_keys = ['Tournament'] + group_keys
        
        if 'Stage' in self.df_validated.columns:
            insert_idx = group_keys.index('Match Name')
            group_keys.insert(insert_idx, 'Stage')
        
        if 'Match Type' in self.df_validated.columns:
            insert_idx = group_keys.index('Match Name')
            group_keys.insert(insert_idx, 'Match Type')
        
        return group_keys
    
    def validate_round_data(self):
        """라운드 데이터 검증"""
        self.log("\n" + "=" * 80)
        self.log("🔍 Phase 1 - Step 2: 라운드 컨텍스트 검증")
        self.log("=" * 80)
        
        group_keys = self.get_group_keys()
        self.log(f"\n📊 그룹화 키: {group_keys}")
        self.log(f"\n📊 각 라운드별 참여 팀 개수 분석:")
        
        round_groups = self.df_validated.groupby(group_keys)
        
        team_counts = []
        suspicious_rounds = []
        
        for group_key, round_data in round_groups:
            # 그룹 키 언팩
            key_dict = {}
            if isinstance(group_key, tuple):
                for i, key_name in enumerate(group_keys):
                    key_dict[key_name] = group_key[i]
            else:
                key_dict[group_keys[0]] = group_key
            
            eliminators = set(round_data['Eliminator Team'].unique())
            eliminated = set(round_data['Eliminated Team'].unique())
            all_teams = eliminators | eliminated
            
            num_teams = len(all_teams)
            total_kills = len(round_data)
            
            team_counts.append({
                'num_teams': num_teams,
                'total_kills': total_kills,
                'teams': list(all_teams),
                **key_dict
            })
            
            if num_teams > 2 or total_kills > 25:
                suspicious_rounds.append({
                    'num_teams': num_teams,
                    'total_kills': total_kills,
                    'teams': list(all_teams),
                    'eliminators': list(eliminators),
                    'eliminated': list(eliminated),
                    **key_dict
                })
        
        # 통계
        if team_counts:
            df_counts = pd.DataFrame(team_counts)
            self.log(f"\n   총 라운드: {len(df_counts)}")
            
            if 'Tournament' in group_keys:
                self.log(f"   총 토너먼트: {df_counts['Tournament'].nunique()}")
            if 'Stage' in group_keys:
                self.log(f"   총 스테이지: {df_counts['Stage'].nunique()}")
            if 'Match Type' in group_keys:
                self.log(f"   총 매치타입: {df_counts['Match Type'].nunique()}")
            
            self.log(f"\n   팀 개수 분포:")
            dist = df_counts['num_teams'].value_counts().sort_index()
            for idx, val in dist.items():
                self.log(f"      {idx}개 팀: {val}개 라운드")
            
            self.log(f"\n   킬 개수 분포:")
            self.log(f"      최소: {df_counts['total_kills'].min()}")
            self.log(f"      평균: {df_counts['total_kills'].mean():.1f}")
            self.log(f"      최대: {df_counts['total_kills'].max()}")
        
        # 비정상 라운드
        if suspicious_rounds:
            self.log(f"\n⚠️ 비정상 감지된 라운드: {len(suspicious_rounds)}개")
            self.log(f"\n   (팀이 2개 초과 또는 킬이 25개 초과)")
            
            for i, sr in enumerate(suspicious_rounds[:20]):
                log_str = f"\n   [{i}]"
                if 'Tournament' in group_keys:
                    log_str += f" Tourn={str(sr['Tournament'])[:20]:20s}"
                if 'Stage' in group_keys:
                    log_str += f" | Stage={str(sr['Stage'])[:15]:15s}"
                if 'Match Type' in group_keys:
                    log_str += f" | Type={str(sr['Match Type'])[:12]:12s}"
                log_str += f" | {sr['Match Name'][:35]}"
                self.log(log_str)
                
                self.log(f"       Map={sr['Map']} | R{sr['Round Number']} | 팀 {sr['num_teams']}개 | 킬 {sr['total_kills']}개")
        else:
            self.log(f"\n✅ 모든 라운드가 정상 (팀 2개, 킬 ≤ 25)")
        
        return len(suspicious_rounds)
    
    def analyze_problematic_rounds(self):
        """10킬 이상 라운드 분석"""
        self.log("\n" + "=" * 80)
        self.log("🔎 Phase 1 - Step 3: 문제 라운드 상세 분석")
        self.log("=" * 80)
        
        group_keys = self.get_group_keys()
        round_groups = self.df_validated.groupby(group_keys)
        
        problematic = []
        
        for group_key, round_data in round_groups:
            key_dict = {}
            if isinstance(group_key, tuple):
                for i, key_name in enumerate(group_keys):
                    key_dict[key_name] = group_key[i]
            else:
                key_dict[group_keys[0]] = group_key
            
            eliminators = round_data['Eliminator Team'].unique()
            eliminated = round_data['Eliminated Team'].unique()
            all_teams = set(eliminators) | set(eliminated)
            
            if len(all_teams) != 2:
                continue
            
            teams = sorted(list(all_teams))
            my_team = teams[0]
            opp_team = teams[1]
            
            my_kills = int((round_data['Eliminator Team'] == my_team).sum())
            opp_kills = int((round_data['Eliminator Team'] == opp_team).sum())
            my_deaths = int((round_data['Eliminated Team'] == my_team).sum())
            opp_deaths = int((round_data['Eliminated Team'] == opp_team).sum())
            
            if my_kills >= 10 or opp_kills >= 10:
                problematic.append({
                    'my_team': my_team,
                    'opp_team': opp_team,
                    'my_kills': my_kills,
                    'opp_kills': opp_kills,
                    'my_deaths': my_deaths,
                    'opp_deaths': opp_deaths,
                    'total_kills': my_kills + opp_kills,
                    'raw_data': round_data,
                    **key_dict
                })
        
        if problematic:
            self.log(f"\n⚠️ 10킬 이상 라운드 발견: {len(problematic)}개\n")
            
            for i, prob in enumerate(problematic[:10]):
                log_str = f"\n   [{i}]"
                if 'Tournament' in group_keys:
                    log_str += f" Tourn: {str(prob['Tournament'])[:25]}"
                if 'Stage' in group_keys:
                    log_str += f" | Stage: {str(prob['Stage'])[:12]}"
                if 'Match Type' in group_keys:
                    log_str += f" | Type: {str(prob['Match Type'])[:10]}"
                self.log(log_str)
                
                self.log(f"       {prob['Map']} Round {prob['Round Number']}")
                self.log(f"       {prob['my_team']}={prob['my_kills']}K vs {prob['opp_team']}={prob['opp_kills']}K (총 {prob['total_kills']}K)")
        else:
            self.log(f"\n✅ 10킬 이상 라운드 없음")
        
        return len(problematic)
    
    # ============================================================
    # 최종 요약
    # ============================================================
    
    def generate_final_summary(self):
        """최종 요약 보고서"""
        self.log("\n" + "=" * 80)
        self.log("📋 최종 요약 보고서")
        self.log("=" * 80)
        
        summary = f"""
✅ Phase 0-1 통합 파이프라인 완료!

📊 Phase 0 - 데이터 중복 제거:
   ├─ 원본 행 수: {len(self.df_original):,}
   ├─ 정제 행 수: {len(self.df_deduped):,}
   ├─ 제거된 행: {len(self.df_original) - len(self.df_deduped):,}
   └─ 저장 위치: {self.deduped_file}

📊 Phase 1 - 라운드 컨텍스트 검증:
   ├─ 검증 행 수: {len(self.df_validated):,}
   ├─ 필수 컬럼: ✅ 모두 확인
   ├─ 데이터 품질: ✅ 검증 완료
   └─ 로그 파일: {self.log_file}

🎯 다음 단계:
   → round_context_features.py 실행 (특성 수집)
      입력 파일: {self.deduped_file}

📁 생성된 파일:
   ├─ {os.path.basename(self.deduped_file)} (정제된 데이터)
   └─ {os.path.basename(self.log_file)} (상세 로그)
        """
        
        self.log(summary)
        return summary


# ============================================================
# 메인 실행
# ============================================================
if __name__ == "__main__":
    processor = DataPipelineProcessor(
        input_file='./data/final/all_matches_cleaned.csv',
        output_dir='./data/final'
    )
    
    import time
    start_time = time.time()
    
    try:
        # 헤더
        processor.log("\n" + "╔" + "=" * 78 + "╗")
        processor.log("║" + " " * 15 + "🔄 Phase 0-1 통합: 데이터 전처리 + 검증" + " " * 20 + "║")
        processor.log("╚" + "=" * 78 + "╝")
        
        # Phase 0: 중복 제거
        if processor.load_original_data():
            num_dups = processor.analyze_duplicates()
            
            if num_dups > 0:
                processor.remove_duplicates()
            else:
                processor.df_deduped = processor.df_original.copy()
            
            processor.save_deduplicated_data()
            
            # Phase 1: 검증
            if processor.prepare_phase1_data():
                suspicious_count = processor.validate_round_data()
                problematic_count = processor.analyze_problematic_rounds()
                
                processor.generate_final_summary()
        
        elapsed = time.time() - start_time
        processor.log(f"\n⏱️  총 소요 시간: {elapsed:.2f}초")
        processor.log("\n✅ 파이프라인 완료!")
        
    except Exception as e:
        processor.log(f"❌ 에러: {str(e)}")
        import traceback
        processor.log(traceback.format_exc())

[2025-12-07 06:04:25] 
╔==============================================================================╗
[2025-12-07 06:04:25] ║               🔄 Phase 0-1 통합: 데이터 전처리 + 검증                    ║
[2025-12-07 06:04:25] ╚==============================================================================╝
[2025-12-07 06:04:25] ================================================================================
[2025-12-07 06:04:25] 📂 Phase 0 - Step 1: 원본 데이터 로드
[2025-12-07 06:04:25] ================================================================================


C:\Users\user\AppData\Local\Temp\ipykernel_28120\361523176.py:44: DtypeWarning: Columns (20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df_original = pd.read_csv(self.input_file)


[2025-12-07 06:04:34] ✅ 파일 로드 완료: ./data/final/all_matches_cleaned.csv
[2025-12-07 06:04:34]    원본 행 수: 3,530,696
[2025-12-07 06:04:34]    컬럼 수: 26
[2025-12-07 06:04:43]    메모리 사용: 5118.6 MB
[2025-12-07 06:04:43] 
[2025-12-07 06:04:43] 🔍 Phase 0 - Step 2: 중복 데이터 분석
[2025-12-07 06:04:43] ================================================================================
[2025-12-07 06:04:48] 
📊 전체 행 중복 분석:
[2025-12-07 06:04:48]    원본 행: 3,530,696
[2025-12-07 06:04:48]    중복 행: 29,360 (0.83%)
[2025-12-07 06:04:48] 
📋 중복 그룹 분석:
[2025-12-07 06:04:48]    중복 그룹 수: 8011
[2025-12-07 06:04:48]    최대 중복 횟수: 16
[2025-12-07 06:04:48]    평균 중복 횟수: 3.7
[2025-12-07 06:04:48] 
   상위 10개 중복 그룹:
[2025-12-07 06:04:48]       16회 중복
[2025-12-07 06:04:48]       16회 중복
[2025-12-07 06:04:48]       16회 중복
[2025-12-07 06:04:48]       16회 중복
[2025-12-07 06:04:48]       16회 중복
[2025-12-07 06:04:48]       16회 중복
[2025-12-07 06:04:48]       16회 중복
[2025-12-07 06:04:48]       16회 중복
[2025-12-07 06:04:48]       16회 중복
[

In [31]:


import pandas as pd
import numpy as np
import os
from datetime import datetime


class DataPipeline:
    """Phase 0-1-2 통합 파이프라인"""
    
    def __init__(self, input_file='./data/final/all_matches_cleaned.csv',
                 output_dir='./data/final'):
        self.input_file = input_file
        self.output_dir = output_dir
        self.deduped_file = os.path.join(output_dir, 'all_matches_deduped.csv')
        self.round_aggregated_file = os.path.join(output_dir, 'round_aggregated.csv')
        self.features_file = os.path.join(output_dir, 'round_with_features.csv')
        os.makedirs(output_dir, exist_ok=True)
        
        self.df_original = None
        self.df_deduped = None
        self.df_validated = None
        self.df_aggregated = None
        self.df_features = None
        self.log_file = os.path.join(output_dir, 'pipeline_phase0_1_2.log')
        
        with open(self.log_file, 'w', encoding='utf-8') as f:
            f.write(f"파이프라인 시작: {datetime.now()}\n")
    
    def log(self, msg):
        """로그 기록"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_msg = f"[{timestamp}] {msg}"
        print(log_msg)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(log_msg + "\n")
    
    # ============================================================
    # PHASE 0: 데이터 중복 제거
    # ============================================================
    
    def phase0_load_data(self):
        """원본 데이터 로드"""
        self.log("=" * 80)
        self.log("📂 Phase 0 - Step 1: 원본 데이터 로드")
        self.log("=" * 80)
        
        try:
            self.df_original = pd.read_csv(self.input_file)
            self.log(f"✅ 파일 로드 완료: {self.input_file}")
            self.log(f"   원본 행 수: {len(self.df_original):,}")
            self.log(f"   컬럼 수: {len(self.df_original.columns)}")
            return True
        except Exception as e:
            self.log(f"❌ 에러: {str(e)}")
            return False
    
    def phase0_remove_duplicates(self):
        """중복 행 제거"""
        self.log("\n" + "=" * 80)
        self.log("🧹 Phase 0 - Step 2: 중복 행 제거")
        self.log("=" * 80)
        
        before_count = len(self.df_original)
        self.df_deduped = self.df_original.drop_duplicates(keep='first').reset_index(drop=True)
        after_count = len(self.df_deduped)
        removed_count = before_count - after_count
        
        self.log(f"\n📊 중복 제거 결과:")
        self.log(f"   제거 전: {before_count:,}")
        self.log(f"   제거 후: {after_count:,}")
        self.log(f"   제거된 행: {removed_count:,} ({removed_count/before_count*100:.2f}%)")
        
        return self.df_deduped
    
    def phase0_save_deduped(self):
        """정제 데이터 저장"""
        try:
            self.df_deduped.to_csv(self.deduped_file, index=False, encoding='utf-8')
            self.log(f"\n💾 정제 데이터 저장: {self.deduped_file}")
            return True
        except Exception as e:
            self.log(f"❌ 저장 에러: {str(e)}")
            return False
    
    # ============================================================
    # PHASE 1: 라운드 집계
    # ============================================================
    
    def phase1_prepare_data(self):
        """Phase 1 데이터 준비"""
        self.df_validated = self.df_deduped.copy()
        
        # Round Number 정수 변환
        self.df_validated['Round Number'] = pd.to_numeric(
            self.df_validated['Round Number'], errors='coerce'
        ).astype('Int64')
        
        nan_count = self.df_validated['Round Number'].isna().sum()
        if nan_count > 0:
            self.log(f"⚠️ Round Number NaN 제거: {nan_count}개")
            self.df_validated = self.df_validated[
                self.df_validated['Round Number'].notna()
            ].reset_index(drop=True)
        
        return True
    
    def get_group_keys(self):
        """그룹화 키 동적 결정"""
        group_keys = ['Match Name', 'Map', 'Round Number']
        
        if 'Tournament' in self.df_validated.columns:
            group_keys = ['Tournament'] + group_keys
        
        if 'Stage' in self.df_validated.columns:
            insert_idx = group_keys.index('Match Name')
            group_keys.insert(insert_idx, 'Stage')
        
        if 'Match Type' in self.df_validated.columns:
            insert_idx = group_keys.index('Match Name')
            group_keys.insert(insert_idx, 'Match Type')
        
        return group_keys
    
    def phase1_aggregate_rounds(self):
        """라운드 집계 (Eliminator Team 기반)"""
        self.log("\n" + "=" * 80)
        self.log("📊 Phase 1 - Step 3: 라운드 집계")
        self.log("=" * 80)
        
        group_keys = self.get_group_keys()
        self.log(f"\n📊 그룹화 키: {group_keys}")
        
        round_groups = self.df_validated.groupby(group_keys)
        aggregated_rows = []
        
        for group_key, round_data in round_groups:
            # 그룹 키 언팩
            key_dict = {}
            if isinstance(group_key, tuple):
                for i, key_name in enumerate(group_keys):
                    key_dict[key_name] = group_key[i]
            else:
                key_dict[group_keys[0]] = group_key
            
            # ✅ Eliminator Team만 사용
            eliminators = set(round_data['Eliminator Team'].unique())
            total_kills = len(round_data)
            
            # 2팀만 처리
            if len(eliminators) == 2:
                teams = sorted(list(eliminators))
                team1 = teams[0]
                team2 = teams[1]
                
                team1_kills = int((round_data['Eliminator Team'] == team1).sum())
                team2_kills = int((round_data['Eliminator Team'] == team2).sum())
                
                team1_deaths = total_kills - team1_kills
                team2_deaths = total_kills - team2_kills
                
                agg_row = {**key_dict}
                agg_row['Team1'] = team1
                agg_row['Team2'] = team2
                agg_row['Team1_Kills'] = team1_kills
                agg_row['Team1_Deaths'] = team1_deaths
                agg_row['Team2_Kills'] = team2_kills
                agg_row['Team2_Deaths'] = team2_deaths
                agg_row['Total_Kills_In_Round'] = total_kills
                
                aggregated_rows.append(agg_row)
        
        self.df_aggregated = pd.DataFrame(aggregated_rows)
        
        self.log(f"\n✅ 라운드 집계 완료:")
        self.log(f"   원본 킬 기록: {len(self.df_validated):,}개")
        self.log(f"   집계된 라운드: {len(self.df_aggregated):,}개")
        self.log(f"   압축율: {len(self.df_validated)/len(self.df_aggregated):.1f}배")
        
        return True
    
    def phase1_save_aggregated(self):
        """집계 데이터 저장"""
        try:
            self.df_aggregated.to_csv(self.round_aggregated_file, index=False, encoding='utf-8')
            self.log(f"\n💾 집계 데이터 저장: {self.round_aggregated_file}")
            return True
        except Exception as e:
            self.log(f"❌ 저장 에러: {str(e)}")
            return False
    
    # ============================================================
    # PHASE 2: 특성 추출
    # ============================================================
    
    def phase2_basic_features(self):
        """라운드 기본 특성 추출"""
        self.log("\n" + "=" * 80)
        self.log("📊 Phase 2 - Step 1: 라운드 기본 특성 추출")
        self.log("=" * 80)
        
        self.df_features = self.df_aggregated.copy()
        
        # ✅ 기본 특성
        self.df_features['Team1_KD_Ratio'] = (
            self.df_features['Team1_Kills'] / 
            (self.df_features['Team1_Deaths'] + 1)  # +1로 나누기 오류 방지
        ).round(2)
        
        self.df_features['Team2_KD_Ratio'] = (
            self.df_features['Team2_Kills'] / 
            (self.df_features['Team2_Deaths'] + 1)
        ).round(2)
        
        self.df_features['Team1_Kill_Efficiency'] = (
            self.df_features['Team1_Kills'] / 
            self.df_features['Total_Kills_In_Round']
        ).round(3)
        
        self.df_features['Team2_Kill_Efficiency'] = (
            self.df_features['Team2_Kills'] / 
            self.df_features['Total_Kills_In_Round']
        ).round(3)
        
        # ✅ 승패
        self.df_features['Team1_Round_Result'] = (
            self.df_features['Team1_Kills'] > self.df_features['Team1_Deaths']
        ).astype(int)  # 1 = Win, 0 = Lose
        
        self.df_features['Team2_Round_Result'] = (
            self.df_features['Team2_Kills'] > self.df_features['Team2_Deaths']
        ).astype(int)
        
        self.log(f"✅ 기본 특성 추출 완료 (4개)")
        return True
    
    def phase2_cumulative_features(self):
        """누적 특성 계산"""
        self.log("\n" + "=" * 80)
        self.log("📊 Phase 2 - Step 2: 누적 특성 계산")
        self.log("=" * 80)
        
        # Match별로 누적 계산
        group_cols = ['Match Name', 'Map']
        if 'Tournament' in self.df_features.columns:
            group_cols = ['Tournament', 'Stage'] + group_cols if 'Stage' in self.df_features.columns else ['Tournament'] + group_cols
        
        for idx, row in self.df_features.iterrows():
            # 같은 경기의 이전 라운드들
            mask = (self.df_features['Match Name'] == row['Match Name']) & \
                   (self.df_features['Round Number'] <= row['Round Number'])
            
            prev_data = self.df_features[mask].iloc[:idx+1]
            
            # Team1 누적
            self.df_features.at[idx, 'Team1_Cumul_Kills'] = prev_data['Team1_Kills'].sum()
            self.df_features.at[idx, 'Team1_Cumul_Deaths'] = prev_data['Team1_Deaths'].sum()
            self.df_features.at[idx, 'Team1_Cumul_Wins'] = prev_data['Team1_Round_Result'].sum()
            
            # Team2 누적
            self.df_features.at[idx, 'Team2_Cumul_Kills'] = prev_data['Team2_Kills'].sum()
            self.df_features.at[idx, 'Team2_Cumul_Deaths'] = prev_data['Team2_Deaths'].sum()
            self.df_features.at[idx, 'Team2_Cumul_Wins'] = prev_data['Team2_Round_Result'].sum()
            
            # 누적 승률
            round_count = len(prev_data)
            self.df_features.at[idx, 'Team1_Win_Rate'] = (
                self.df_features.at[idx, 'Team1_Cumul_Wins'] / round_count * 100
            ) if round_count > 0 else 0
            
            self.df_features.at[idx, 'Team2_Win_Rate'] = (
                self.df_features.at[idx, 'Team2_Cumul_Wins'] / round_count * 100
            ) if round_count > 0 else 0
        
        self.log(f"✅ 누적 특성 계산 완료 (6개)")
        return True
    
    def phase2_momentum_features(self):
        """모멘텀 & 추세 특성"""
        self.log("\n" + "=" * 80)
        self.log("📊 Phase 2 - Step 3: 모멘텀 & 추세 특성")
        self.log("=" * 80)
        
        for idx, row in self.df_features.iterrows():
            mask = (self.df_features['Match Name'] == row['Match Name']) & \
                   (self.df_features['Round Number'] <= row['Round Number'])
            
            prev_data = self.df_features[mask].iloc[max(0, idx-2):idx+1]  # 최근 3라운드
            
            # Team1 최근 승률
            if len(prev_data) > 0:
                self.df_features.at[idx, 'Team1_Recent3_WinRate'] = (
                    prev_data['Team1_Round_Result'].sum() / len(prev_data) * 100
                )
                self.df_features.at[idx, 'Team2_Recent3_WinRate'] = (
                    prev_data['Team2_Round_Result'].sum() / len(prev_data) * 100
                )
            
            # 최근 라운드 결과
            if idx > 0:
                self.df_features.at[idx, 'Team1_Last_Round_Result'] = (
                    self.df_features.iloc[idx-1]['Team1_Round_Result']
                )
                self.df_features.at[idx, 'Team2_Last_Round_Result'] = (
                    self.df_features.iloc[idx-1]['Team2_Round_Result']
                )
        
        self.log(f"✅ 모멘텀 특성 계산 완료 (4개)")
        return True
    
    def phase2_map_features(self):
        """맵별 특성"""
        self.log("\n" + "=" * 80)
        self.log("📊 Phase 2 - Step 4: 맵별 특성")
        self.log("=" * 80)
        
        for idx, row in self.df_features.iterrows():
            current_map = row['Map']
            current_match = row['Match Name']
            
            # 같은 경기, 같은 맵의 이전 라운드
            mask = (self.df_features['Match Name'] == current_match) & \
                   (self.df_features['Map'] == current_map) & \
                   (self.df_features['Round Number'] < row['Round Number'])
            
            prev_map_data = self.df_features[mask]
            
            if len(prev_map_data) > 0:
                # Team1 맵별 승률
                self.df_features.at[idx, 'Team1_Map_WinRate'] = (
                    prev_map_data['Team1_Round_Result'].sum() / len(prev_map_data) * 100
                )
                self.df_features.at[idx, 'Team2_Map_WinRate'] = (
                    prev_map_data['Team2_Round_Result'].sum() / len(prev_map_data) * 100
                )
        
        self.log(f"✅ 맵별 특성 계산 완료 (2개)")
        return True
    
    def phase2_h2h_features(self):
        """상성(Head-to-Head) 특성"""
        self.log("\n" + "=" * 80)
        self.log("📊 Phase 2 - Step 5: 상성 특성")
        self.log("=" * 80)
        
        for idx, row in self.df_features.iterrows():
            team1 = row['Team1']
            team2 = row['Team2']
            
            # 같은 팀 조합의 이전 경기
            mask = (
                ((self.df_features['Team1'] == team1) & (self.df_features['Team2'] == team2)) |
                ((self.df_features['Team1'] == team2) & (self.df_features['Team2'] == team1))
            ) & (self.df_features.index < idx)
            
            prev_h2h = self.df_features[mask]
            
            if len(prev_h2h) > 0:
                # Team1 상성 승률
                team1_wins = ((prev_h2h['Team1'] == team1) & (prev_h2h['Team1_Round_Result'] == 1)).sum() + \
                            ((prev_h2h['Team2'] == team1) & (prev_h2h['Team2_Round_Result'] == 1)).sum()
                
                self.df_features.at[idx, 'Team1_H2H_WinRate'] = (
                    team1_wins / len(prev_h2h) * 100
                )
                
                team2_wins = ((prev_h2h['Team1'] == team2) & (prev_h2h['Team1_Round_Result'] == 1)).sum() + \
                            ((prev_h2h['Team2'] == team2) & (prev_h2h['Team2_Round_Result'] == 1)).sum()
                
                self.df_features.at[idx, 'Team2_H2H_WinRate'] = (
                    team2_wins / len(prev_h2h) * 100
                )
        
        self.log(f"✅ 상성 특성 계산 완료 (2개)")
        return True
    
    def phase2_save_features(self):
        """특성 데이터 저장"""
        try:
            # NaN 값을 0으로 채우기
            self.df_features = self.df_features.fillna(0)
            
            self.df_features.to_csv(self.features_file, index=False, encoding='utf-8')
            self.log(f"\n💾 특성 데이터 저장: {self.features_file}")
            self.log(f"   행: {len(self.df_features):,}")
            self.log(f"   컬럼: {len(self.df_features.columns)}")
            return True
        except Exception as e:
            self.log(f"❌ 저장 에러: {str(e)}")
            return False
    
    # ============================================================
    # 최종 요약
    # ============================================================
    
    def generate_summary(self):
        """최종 요약 보고서"""
        self.log("\n" + "=" * 80)
        self.log("📋 최종 요약 보고서")
        self.log("=" * 80)
        
        summary = f"""
✅ Phase 0-1-2 통합 파이프라인 완료!

📊 Phase 0 - 데이터 중복 제거:
   ├─ 원본 행 수: {len(self.df_original):,}
   ├─ 정제 행 수: {len(self.df_deduped):,}
   └─ 제거된 행: {len(self.df_original) - len(self.df_deduped):,}

📊 Phase 1 - 라운드 집계:
   ├─ 원본 킬 기록: {len(self.df_validated):,}
   ├─ 집계된 라운드: {len(self.df_aggregated):,}
   └─ 압축율: {len(self.df_validated)/len(self.df_aggregated):.1f}배

📊 Phase 2 - 특성 추출:
   ├─ 생성된 특성: {len(self.df_features.columns) - len(self.df_aggregated.columns)}개
   ├─ 최종 컬럼: {len(self.df_features.columns)}개
   └─ 최종 행: {len(self.df_features):,}개

🎯 생성된 파일:
   ├─ {os.path.basename(self.deduped_file)} (Phase 0)
   ├─ {os.path.basename(self.round_aggregated_file)} (Phase 1)
   ├─ {os.path.basename(self.features_file)} (Phase 2) ⭐
   └─ {os.path.basename(self.log_file)} (로그)

🚀 다음 단계:
   → 모델 학습 (XGBoost / LSTM)
   → 경기 승패 예측
   → 라운드별 성적 예측
        """
        
        self.log(summary)
        return summary
    
    def run(self):
        """전체 파이프라인 실행"""
        self.log("\n" + "╔" + "=" * 78 + "╗")
        self.log("║" + " " * 10 + "🔄 Phase 0-1-2 통합: 전처리 + 검증 + 특성추출" + " " * 23 + "║")
        self.log("╚" + "=" * 78 + "╝")
        
        # Phase 0
        if not self.phase0_load_data():
            return False
        
        self.phase0_remove_duplicates()
        if not self.phase0_save_deduped():
            return False
        
        # Phase 1
        self.phase1_prepare_data()
        self.phase1_aggregate_rounds()
        if not self.phase1_save_aggregated():
            return False
        
        # Phase 2
        self.phase2_basic_features()
        self.phase2_cumulative_features()
        self.phase2_momentum_features()
        self.phase2_map_features()
        self.phase2_h2h_features()
        if not self.phase2_save_features():
            return False
        
        # 최종 요약
        self.generate_summary()
        
        return True


# ============================================================
# 메인 실행
# ============================================================
if __name__ == "__main__":
    import time
    
    pipeline = DataPipeline(
        input_file='./data/final/all_matches_cleaned.csv',
        output_dir='./data/final'
    )
    
    start_time = time.time()
    
    try:
        success = pipeline.run()
        
        elapsed = time.time() - start_time
        
        if success:
            pipeline.log(f"\n⏱️  총 소요 시간: {elapsed:.2f}초")
            pipeline.log("\n✅ 파이프라인 완료!")
        else:
            pipeline.log("\n❌ 파이프라인 실패!")
    
    except Exception as e:
        pipeline.log(f"❌ 예외 발생: {str(e)}")
        import traceback
        pipeline.log(traceback.format_exc())

[2025-12-07 06:27:02] 
╔==============================================================================╗
[2025-12-07 06:27:02] ║          🔄 Phase 0-1-2 통합: 전처리 + 검증 + 특성추출                       ║
[2025-12-07 06:27:02] ╚==============================================================================╝
[2025-12-07 06:27:02] ================================================================================
[2025-12-07 06:27:02] 📂 Phase 0 - Step 1: 원본 데이터 로드
[2025-12-07 06:27:02] ================================================================================


C:\Users\user\AppData\Local\Temp\ipykernel_28120\556955346.py:48: DtypeWarning: Columns (20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df_original = pd.read_csv(self.input_file)


[2025-12-07 06:27:11] ✅ 파일 로드 완료: ./data/final/all_matches_cleaned.csv
[2025-12-07 06:27:11]    원본 행 수: 3,530,696
[2025-12-07 06:27:11]    컬럼 수: 26
[2025-12-07 06:27:11] 
[2025-12-07 06:27:11] 🧹 Phase 0 - Step 2: 중복 행 제거
[2025-12-07 06:27:11] ================================================================================
[2025-12-07 06:27:17] 
📊 중복 제거 결과:
[2025-12-07 06:27:17]    제거 전: 3,530,696
[2025-12-07 06:27:17]    제거 후: 3,509,347
[2025-12-07 06:27:17]    제거된 행: 21,349 (0.60%)
[2025-12-07 06:27:38] 
💾 정제 데이터 저장: ./data/final\all_matches_deduped.csv
[2025-12-07 06:27:38] ⚠️ Round Number NaN 제거: 6163개
[2025-12-07 06:27:40] 
[2025-12-07 06:27:40] 📊 Phase 1 - Step 3: 라운드 집계
[2025-12-07 06:27:40] ================================================================================
[2025-12-07 06:27:40] 
📊 그룹화 키: ['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map', 'Round Number']
[2025-12-07 06:28:21] 
✅ 라운드 집계 완료:
[2025-12-07 06:28:21]    원본 킬 기록: 3,503,184개
[2025-12-07 06:28:21]   

KeyboardInterrupt: 

In [33]:

import pandas as pd
import numpy as np
import os
from datetime import datetime


class SequenceFeatureExtractor:
    """Seq2Seq 모델용 시퀀스 특성 추출"""
    
    def __init__(self, input_file='./data/final/round_aggregated.csv',
                 output_dir='./data/final'):
        self.input_file = input_file
        self.output_dir = output_dir
        self.sequence_file = os.path.join(output_dir, 'round_sequence_features.csv')
        os.makedirs(output_dir, exist_ok=True)
        
        self.df_rounds = None
        self.df_sequence = None
        self.log_file = os.path.join(output_dir, 'feature_extraction.log')
        
        with open(self.log_file, 'w', encoding='utf-8') as f:
            f.write(f"특성 추출 시작: {datetime.now()}\n")
    
    def log(self, msg):
        """로그 기록"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_msg = f"[{timestamp}] {msg}"
        print(log_msg)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(log_msg + "\n")
    
    # ============================================================
    # Step 1: 데이터 로드 및 정렬
    # ============================================================
    
    def step1_load_and_sort(self):
        """라운드 데이터 로드 및 정렬"""
        self.log("=" * 80)
        self.log("📂 Step 1: 데이터 로드 및 정렬")
        self.log("=" * 80)
        
        try:
            self.df_rounds = pd.read_csv(self.input_file)
            self.log(f"✅ 파일 로드: {self.input_file}")
            self.log(f"   행: {len(self.df_rounds):,}, 컬럼: {len(self.df_rounds.columns)}")
            
            # 정렬
            sort_cols = ['Match Name', 'Map', 'Round Number']
            self.df_rounds = self.df_rounds.sort_values(sort_cols).reset_index(drop=True)
            self.log(f"✅ 정렬 완료: {sort_cols}")
            
            return True
        except Exception as e:
            self.log(f"❌ 에러: {str(e)}")
            return False
    
    # ============================================================
    # Step 2: 라운드 결과 추출
    # ============================================================
    
    def step2_round_outcomes(self):
        """라운드 승패 추출"""
        self.log("\n" + "=" * 80)
        self.log("🏆 Step 2: 라운드 결과 추출")
        self.log("=" * 80)
        
        # 라운드 인덱스 (경기별)
        self.df_rounds['Round_Index'] = (
            self.df_rounds.groupby(['Match Name', 'Map']).cumcount()
        )
        
        # 라운드 승패 (Team1 기준)
        self.df_rounds['Team1_Won'] = (
            self.df_rounds['Team1_Kills'] > self.df_rounds['Team1_Deaths']
        ).astype(int)
        
        self.df_rounds['Team2_Won'] = (
            self.df_rounds['Team2_Kills'] > self.df_rounds['Team2_Deaths']
        ).astype(int)
        
        self.log(f"✅ 라운드 결과 추출 완료")
        self.log(f"   Round_Index: 경기별 라운드 순서")
        self.log(f"   Team1_Won: 0 또는 1")
        
        return True
    
    # ============================================================
    # Step 3: 누적 점수 계산 (My Score, Opp Score)
    # ============================================================
    
    def step3_cumulative_score(self):
        """누적 점수 계산"""
        self.log("\n" + "=" * 80)
        self.log("📊 Step 3: 누적 점수 계산")
        self.log("=" * 80)
        
        # Team1 기준
        self.df_rounds['Team1_Cumul_Wins'] = (
            self.df_rounds.groupby(['Match Name', 'Map'])['Team1_Won'].cumsum()
        )
        
        self.df_rounds['Team1_Cumul_Losses'] = (
            self.df_rounds.groupby(['Match Name', 'Map'])['Team2_Won'].cumsum()
        )
        
        # Team2 기준
        self.df_rounds['Team2_Cumul_Wins'] = (
            self.df_rounds.groupby(['Match Name', 'Map'])['Team2_Won'].cumsum()
        )
        
        self.df_rounds['Team2_Cumul_Losses'] = (
            self.df_rounds.groupby(['Match Name', 'Map'])['Team1_Won'].cumsum()
        )
        
        self.log(f"✅ 누적 점수 계산 완료")
        self.log(f"   Team1_Cumul_Wins: 아군 누적 승리")
        self.log(f"   Team1_Cumul_Losses: 적군 누적 승리")
        
        return True
    
    # ============================================================
    # Step 4: 경제 특성 계산 (My Bank, Loadout Value)
    # ============================================================
    
    def step4_economy_features(self):
        """경제 특성 계산"""
        self.log("\n" + "=" * 80)
        self.log("💰 Step 4: 경제 특성 계산")
        self.log("=" * 80)
        
        # VALORANT 경제 규칙
        KILL_REWARD = 100       # 킬 보상
        DEATH_PENALTY = 0       # 데스 페널티 (데스 자체는 페널티 없음)
        ROUND_WIN_REWARD = 3900 # 라운드 승리 보상
        ROUND_LOSS_REWARD = 2400  # 라운드 패배 보상 (일정 라운드 연패 후)
        
        for match_group_name, match_group in self.df_rounds.groupby(['Match Name', 'Map']):
            indices = match_group.index
            
            for i, idx in enumerate(indices):
                # Team1 경제
                if i == 0:
                    # 첫 라운드: 초기 자금 2400
                    team1_bank = 2400
                    team2_bank = 2400
                else:
                    prev_idx = indices[i - 1]
                    team1_bank = self.df_rounds.at[prev_idx, 'Team1_Bank']
                    team2_bank = self.df_rounds.at[prev_idx, 'Team2_Bank']
                
                # 킬 보상
                team1_kills = self.df_rounds.at[idx, 'Team1_Kills']
                team1_deaths = self.df_rounds.at[idx, 'Team1_Deaths']
                team2_kills = self.df_rounds.at[idx, 'Team2_Kills']
                team2_deaths = self.df_rounds.at[idx, 'Team2_Deaths']
                
                team1_kill_reward = team1_kills * KILL_REWARD
                team2_kill_reward = team2_kills * KILL_REWARD
                
                # 라운드 결과 보상
                team1_won = self.df_rounds.at[idx, 'Team1_Won']
                team2_won = self.df_rounds.at[idx, 'Team2_Won']
                
                team1_round_reward = ROUND_WIN_REWARD if team1_won else ROUND_LOSS_REWARD
                team2_round_reward = ROUND_WIN_REWARD if team2_won else ROUND_LOSS_REWARD
                
                # 누적 자금 (가상 추정치)
                team1_bank = team1_bank + team1_kill_reward + team1_round_reward
                team2_bank = team2_bank + team2_kill_reward + team2_round_reward
                
                self.df_rounds.at[idx, 'Team1_Bank'] = min(team1_bank, 150000)  # 최대 자금 제한
                self.df_rounds.at[idx, 'Team2_Bank'] = min(team2_bank, 150000)
                
                # Loadout Value (현재 라운드 무기 투자액 추정)
                # 킬/데스 비율 기반: 우위 팀이 더 많이 투자
                if team1_kills + team2_kills > 0:
                    team1_kill_ratio = team1_kills / (team1_kills + team2_kills)
                    team1_loadout = 4000 + (team1_kill_ratio * 4000)  # 4000~8000
                    team2_loadout = 4000 + ((1 - team1_kill_ratio) * 4000)
                else:
                    team1_loadout = 2400  # 기본값
                    team2_loadout = 2400
                
                self.df_rounds.at[idx, 'Team1_Loadout'] = round(team1_loadout, 2)
                self.df_rounds.at[idx, 'Team2_Loadout'] = round(team2_loadout, 2)
        
        self.log(f"✅ 경제 특성 계산 완료")
        self.log(f"   Team1_Bank: 누적 자금")
        self.log(f"   Team1_Loadout: 장비 투자액")
        
        return True
    
    # ============================================================
    # Step 5: 구매 전략 분류 (Buy Type)
    # ============================================================
    
    def step5_buy_type_classification(self):
        """구매 전략 분류"""
        self.log("\n" + "=" * 80)
        self.log("🛒 Step 5: 구매 전략 분류")
        self.log("=" * 80)
        
        def classify_buy_type(loadout_value, bank):
            """Loadout Value와 Bank 기반 Buy Type 분류"""
            if loadout_value < 2000:
                return 'Eco'  # 경제 절약
            elif loadout_value < 4000:
                return 'Half-Buy'  # 반쪽 구매
            elif loadout_value >= 7000:
                return 'Full-Buy'  # 풀 구매
            else:
                return 'Save'  # 자금 절약
        
        self.df_rounds['Team1_Buy_Type'] = self.df_rounds.apply(
            lambda row: classify_buy_type(row['Team1_Loadout'], row['Team1_Bank']),
            axis=1
        )
        
        self.df_rounds['Team2_Buy_Type'] = self.df_rounds.apply(
            lambda row: classify_buy_type(row['Team2_Loadout'], row['Team2_Bank']),
            axis=1
        )
        
        self.log(f"✅ 구매 전략 분류 완료")
        self.log(f"   Buy Type: Eco / Half-Buy / Full-Buy / Save")
        
        return True
    
    # ============================================================
    # Step 6: 최종 시퀀스 테이블 생성 (킬수 특성 추가)
    # ============================================================
    
    def step6_create_sequence_table(self):
        """최종 시퀀스 테이블 생성 (킬수 포함)"""
        self.log("\n" + "=" * 80)
        self.log("📋 Step 6: 최종 시퀀스 테이블 생성 (킬수 포함)")
        self.log("=" * 80)
        
        # Team1 기준 시퀀스 테이블
        sequence_features = pd.DataFrame({
            'Tournament': self.df_rounds.get('Tournament', ''),
            'Stage': self.df_rounds.get('Stage', ''),
            'Match_Name': self.df_rounds['Match Name'],
            'Map': self.df_rounds['Map'],
            'Round_Index': self.df_rounds['Round_Index'],
            
            # 특성: My (Team1 기준)
            'My_Score': self.df_rounds['Team1_Cumul_Wins'],
            'Opp_Score': self.df_rounds['Team1_Cumul_Losses'],
            'My_Kills': self.df_rounds['Team1_Kills'],  # ✅ 추가
            'Opp_Kills': self.df_rounds['Team2_Kills'],  # ✅ 추가
            'My_Bank': self.df_rounds['Team1_Bank'],
            'My_Loadout_Value': self.df_rounds['Team1_Loadout'],
            'Buy_Type': self.df_rounds['Team1_Buy_Type'],
            'Round_Outcome': self.df_rounds['Team1_Won'],
            
            # Team1 원본 정보 (디버깅용)
            'Team1': self.df_rounds['Team1'],
            'Team2': self.df_rounds['Team2'],
            'Team1_Kills': self.df_rounds['Team1_Kills'],
            'Team1_Deaths': self.df_rounds['Team1_Deaths'],
            'Team2_Kills': self.df_rounds['Team2_Kills'],
            'Team2_Deaths': self.df_rounds['Team2_Deaths'],
        })
        
        # NaN 처리
        sequence_features = sequence_features.fillna(0)
        
        self.df_sequence = sequence_features
        
        self.log(f"✅ 최종 시퀀스 테이블 생성 완료")
        self.log(f"   행: {len(self.df_sequence):,}")
        self.log(f"   컬럼: {len(self.df_sequence.columns)}")
        self.log(f"   ✅ My_Kills & Opp_Kills 특성 추가됨")
        
        return True
    
    # ============================================================
    # Step 7: 저장 및 검증
    # ============================================================
    
    def step7_save_and_validate(self):
        """특성 저장 및 검증"""
        self.log("\n" + "=" * 80)
        self.log("💾 Step 7: 저장 및 검증")
        self.log("=" * 80)
        
        try:
            self.df_sequence.to_csv(self.sequence_file, index=False, encoding='utf-8')
            self.log(f"✅ 파일 저장: {self.sequence_file}")
            
            # 검증
            self.log(f"\n📊 특성 검증:")
            
            # 특성별 범위 확인
            features_to_check = {
                'My_Score': (0, 100),
                'Opp_Score': (0, 100),
                'My_Kills': (0, 10),  # ✅ 추가
                'Opp_Kills': (0, 10),  # ✅ 추가
                'My_Bank': (0, 150000),
                'My_Loadout_Value': (0, 10000),
                'Round_Outcome': (0, 1),
            }
            
            for feature, (min_val, max_val) in features_to_check.items():
                actual_min = self.df_sequence[feature].min()
                actual_max = self.df_sequence[feature].max()
                avg_val = self.df_sequence[feature].mean()
                self.log(f"   {feature}: {actual_min:.0f} ~ {actual_max:.0f} (평균: {avg_val:.2f})")
            
            # Buy Type 분포
            buy_type_dist = self.df_sequence['Buy_Type'].value_counts()
            self.log(f"\n📊 Buy Type 분포:")
            for buy_type, count in buy_type_dist.items():
                pct = count / len(self.df_sequence) * 100
                self.log(f"   {buy_type}: {count:,} ({pct:.1f}%)")
            
            return True
        except Exception as e:
            self.log(f"❌ 저장 에러: {str(e)}")
            return False
    
    # ============================================================
    # Step 8: 최종 요약
    # ============================================================
    
    def step8_summary(self):
        """최종 요약"""
        self.log("\n" + "=" * 80)
        self.log("📋 최종 요약")
        self.log("=" * 80)
        
        summary = f"""
✅ 시퀀스 특성 추출 완료! (킬수 특성 포함)

📊 데이터 통계:
   ├─ 총 라운드: {len(self.df_sequence):,}개
   ├─ 경기 수: {self.df_sequence['Match_Name'].nunique()}개
   ├─ 맵 종류: {self.df_sequence['Map'].nunique()}개
   └─ 라운드 인덱스 범위: {self.df_sequence['Round_Index'].min():.0f}~{self.df_sequence['Round_Index'].max():.0f}

📋 특성 목록 (9개):
   1️⃣ Round Index: 시퀀스 내 라운드 순서
   2️⃣ My Score: 누적 점수 (평균: {self.df_sequence['My_Score'].mean():.1f})
   3️⃣ Opp Score: 상대 누적 점수 (평균: {self.df_sequence['Opp_Score'].mean():.1f})
   4️⃣ My Kills: 현재 라운드 킬수 (평균: {self.df_sequence['My_Kills'].mean():.2f}) ✅
   5️⃣ Opp Kills: 상대 현재 라운드 킬수 (평균: {self.df_sequence['Opp_Kills'].mean():.2f}) ✅
   6️⃣ My Bank: 누적 자금 (평균: {self.df_sequence['My_Bank'].mean():.0f})
   7️⃣ My Loadout Value: 장비 투자액 (평균: {self.df_sequence['My_Loadout_Value'].mean():.0f})
   8️⃣ Buy Type: 구매 전략 ({self.df_sequence['Buy_Type'].nunique()}가지)
   9️⃣ Round Outcome: 라운드 승패 (승률: {self.df_sequence['Round_Outcome'].mean()*100:.1f}%)

🎯 사용 방식:
   입력: 라운드 1~T의 특성들 (Encoder)
   출력: 라운드 T+1의 특성들 예측 (Decoder)

📁 생성 파일:
   {os.path.basename(self.sequence_file)} ⭐
        """
        
        self.log(summary)
        return summary
    
    def run(self):
        """전체 특성 추출 파이프라인"""
        self.log("\n" + "╔" + "=" * 78 + "╗")
        self.log("║" + " " * 15 + "🔄 시퀀스 특성 추출 (Seq2Seq 모델용)" + " " * 23 + "║")
        self.log("╚" + "=" * 78 + "╝")
        
        steps = [
            self.step1_load_and_sort,
            self.step2_round_outcomes,
            self.step3_cumulative_score,
            self.step4_economy_features,
            self.step5_buy_type_classification,
            self.step6_create_sequence_table,
            self.step7_save_and_validate,
            self.step8_summary,
        ]
        
        for step in steps:
            if not step():
                self.log(f"\n❌ {step.__name__} 실패!")
                return False
        
        return True


# ============================================================
# 메인 실행
# ============================================================
if __name__ == "__main__":
    import time
    
    extractor = SequenceFeatureExtractor(
        input_file='./data/final/round_aggregated.csv',
        output_dir='./data/final'
    )
    
    start_time = time.time()
    
    try:
        success = extractor.run()
        
        elapsed = time.time() - start_time
        
        if success:
            extractor.log(f"\n⏱️  총 소요 시간: {elapsed:.2f}초")
            extractor.log("\n✅ 특성 추출 완료!")
        else:
            extractor.log("\n❌ 특성 추출 실패!")
    
    except Exception as e:
        extractor.log(f"❌ 예외 발생: {str(e)}")
        import traceback
        extractor.log(traceback.format_exc())

[2025-12-07 06:43:08] 
╔==============================================================================╗
[2025-12-07 06:43:08] ║               🔄 시퀀스 특성 추출 (Seq2Seq 모델용)                       ║
[2025-12-07 06:43:08] ╚==============================================================================╝
[2025-12-07 06:43:08] ================================================================================
[2025-12-07 06:43:08] 📂 Step 1: 데이터 로드 및 정렬
[2025-12-07 06:43:08] ================================================================================
[2025-12-07 06:43:09] ✅ 파일 로드: ./data/final/round_aggregated.csv
[2025-12-07 06:43:09]    행: 153,359, 컬럼: 13
[2025-12-07 06:43:09] ✅ 정렬 완료: ['Match Name', 'Map', 'Round Number']
[2025-12-07 06:43:09] 
[2025-12-07 06:43:09] 🏆 Step 2: 라운드 결과 추출
[2025-12-07 06:43:09] ================================================================================
[2025-12-07 06:43:09] ✅ 라운드 결과 추출 완료
[2025-12-07 06:43:09]    Round_Index: 경기별 라운드 순서
[2025-12-07 06:43:09]  

In [1]:

import pandas as pd
import numpy as np
import os
from datetime import datetime


class Phase2DualPerspectiveAggregation:
    """Phase 2 개선: 양쪽 팀 관점 모두 포함 (Team A 기준 + Team B 기준)"""
    
    def __init__(self, merged_file='./data/final/all_matches_deduped.csv',
                 output_dir='./data/final'):
        self.merged_file = merged_file
        self.output_dir = output_dir
        self.output_file = os.path.join(output_dir, 'round_aggregated_dual_perspective.csv')
        self.log_file = os.path.join(output_dir, 'phase2_dual_perspective.log')
        
        os.makedirs(output_dir, exist_ok=True)
        
        with open(self.log_file, 'w', encoding='utf-8') as f:
            f.write(f"Phase 2 Dual Perspective: {datetime.now()}\n")
    
    def log(self, msg):
        """로그 출력 및 저장"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_msg = f"[{timestamp}] {msg}"
        print(log_msg)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(log_msg + "\n")
    
    def parse_currency(self, val):
        """통화 문자열을 숫자로 변환"""
        if pd.isna(val):
            return 0.0
        if isinstance(val, (int, float)):
            return float(val)
        
        val = str(val).strip()
        try:
            if 'k' in val.lower():
                val = val.replace('k', '').replace('K', '')
                num = float(val) * 1000
            else:
                num = float(val)
            return num
        except:
            return 0.0
    
    def encode_buy_type(self, val):
        """구매 타입 인코딩"""
        if pd.isna(val):
            return 0
        val = str(val).lower()
        if 'full' in val:
            return 2
        elif 'half' in val or 'semi' in val or 'force' in val:
            return 1
        else:  # eco
            return 0
    
    def aggregate_rounds(self):
        """양쪽 팀 관점 모두 포함하여 라운드 집계"""
        self.log("=" * 80)
        self.log("📊 라운드 집계 (양쪽 팀 관점 포함)")
        self.log("=" * 80)
        
        try:
            # ============================================================
            # 1️⃣ 데이터 로드 및 기본 전처리
            # ============================================================
            df = pd.read_csv(self.merged_file)
            self.log(f"\n✅ 데이터 로드: {len(df):,}행 × {len(df.columns)}컬럼")
            
            # 컬럼 확인
            self.log(f"\n🔍 사용 가능한 컬럼:")
            print("\n모든 컬럼:")
            for i, col in enumerate(df.columns, 1):
                print(f"  {i:2d}. {col}")
            
            # Round Number 변환
            if 'Round Number' not in df.columns:
                self.log("❌ Round Number 없음!")
                return False
            
            df['Round Number'] = pd.to_numeric(df['Round Number'], errors='coerce').fillna(0).astype(int)
            df = df[df['Round Number'] > 0]
            self.log(f"✅ Round Number: 변환 완료")
            
            # 통화 컬럼 정제
            for col in ['Loadout Value', 'Remaining Credits']:
                if col in df.columns:
                    df[col] = df[col].apply(self.parse_currency)
                    self.log(f"✅ {col}: 정제 완료")
            
            # Buy Type 인코딩
            if 'Type' in df.columns:
                df['Buy Type Encoded'] = df['Type'].apply(self.encode_buy_type)
                self.log(f"✅ Buy Type: 인코딩 완료")
            
            # ============================================================
            # 2️⃣ 라운드 결과 정의 (핵심!)
            # ============================================================
            df['Team_A_Win'] = 0
            if 'Team Ascore' in df.columns and 'Team Bscore' in df.columns:
                df['Team_A_Win'] = (df['Team Ascore'] > df['Team Bscore']).astype(int)
                self.log(f"✅ Round Outcome (Team A Win): 생성 완료")
            
            # Team B 승리 = Team A 승리의 반대
            df['Team_B_Win'] = 1 - df['Team_A_Win']
            self.log(f"✅ Round Outcome (Team B Win): 생성 완료")
            
            # ============================================================
            # 3️⃣ Team A 관점 집계
            # ============================================================
            self.log(f"\n\n{'='*80}")
            self.log("🔵 Team A 관점 집계 중...")
            self.log(f"{'='*80}")
            
            df_team_a = df.copy()
            self._prepare_perspective(df_team_a, 'Team A', 'Team B', 'Team_A_Win')
            df_aggregated_a = self._aggregate_by_perspective(df_team_a, 'Team A')
            self.log(f"✅ Team A 관점 집계: {len(df_aggregated_a):,}행")
            
            # ============================================================
            # 4️⃣ Team B 관점 집계
            # ============================================================
            self.log(f"\n\n{'='*80}")
            self.log("🟠 Team B 관점 집계 중...")
            self.log(f"{'='*80}")
            
            df_team_b = df.copy()
            self._prepare_perspective(df_team_b, 'Team B', 'Team A', 'Team_B_Win')
            df_aggregated_b = self._aggregate_by_perspective(df_team_b, 'Team B')
            self.log(f"✅ Team B 관점 집계: {len(df_aggregated_b):,}행")
            
            # ============================================================
            # 5️⃣ 최종 병합
            # ============================================================
            self.log(f"\n\n{'='*80}")
            self.log("🔀 최종 병합 중...")
            self.log(f"{'='*80}")
            
            df_final = pd.concat([df_aggregated_a, df_aggregated_b], 
                                  ignore_index=True, sort=False)
            
            self.log(f"\n✅ 최종 병합:")
            self.log(f"   Team A 관점: {len(df_aggregated_a):,}행")
            self.log(f"   Team B 관점: {len(df_aggregated_b):,}행")
            self.log(f"   총합: {len(df_final):,}행")
            self.log(f"   컬럼: {len(df_final.columns)}")
            
            # ============================================================
            # 6️⃣ 최종 컬럼 정렬 및 저장
            # ============================================================
            final_cols = [
                'Tournament', 'Stage', 'Match Name', 'Map', 'Round Number',
                'My Team', 'Opponent Team', 'My Score', 'Opp Score',
                'Year', 'Round Win'
            ]
            
            # 기존 컬럼들 추가
            for col in df_final.columns:
                if col not in final_cols:
                    final_cols.append(col)
            
            df_final = df_final[final_cols]
            
            # CSV 저장
            df_final.to_csv(self.output_file, index=False, encoding='utf-8')
            self.log(f"\n✅ 저장: {self.output_file}")
            
            # ============================================================
            # 7️⃣ 최종 정보 출력
            # ============================================================
            self.log(f"\n📊 최종 컬럼 ({len(df_final.columns)}개):")
            for i, col in enumerate(df_final.columns, 1):
                print(f"  {i:2d}. {col}")
            
            self.log(f"\n📈 최종 데이터 요약:")
            self.log(f"   Shape: {df_final.shape}")
            self.log(f"   Memory: {df_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
            
            # ============================================================
            # 8️⃣ 데이터 검증
            # ============================================================
            self.log(f"\n🔍 데이터 검증:")
            team_a_count = (df_final['My Team'] == 'Team A').sum()
            team_b_count = (df_final['My Team'] == 'Team B').sum()
            self.log(f"   Team A 관점 행: {team_a_count:,}")
            self.log(f"   Team B 관점 행: {team_b_count:,}")
            self.log(f"   균형: {team_a_count == team_b_count} ✅" if team_a_count == team_b_count else f"   균형: False ❌")
            
            win_ratio = df_final['Round Win'].sum() / len(df_final)
            self.log(f"   전체 승률: {win_ratio:.1%}")
            self.log(f"   (이상적: ~50%)")
            
            return True
        
        except Exception as e:
            self.log(f"❌ 에러: {str(e)}")
            import traceback
            self.log(traceback.format_exc())
            return False
    
    def _prepare_perspective(self, df, my_team_name, opp_team_name, win_col):
        """특정 팀 관점으로 컬럼 준비"""
        
        # My Team vs Opponent Team 컬럼명 매핑
        if my_team_name == 'Team A':
            my_score_col = 'Team Ascore'
            opp_score_col = 'Team Bscore'
            my_scoremap_col = 'Team A Scoremap'
            opp_scoremap_col = 'Team B Scoremap'
        else:  # Team B
            my_score_col = 'Team Bscore'
            opp_score_col = 'Team Ascore'
            my_scoremap_col = 'Team B Scoremap'
            opp_scoremap_col = 'Team A Scoremap'
        
        # 관점 컬럼 추가
        df['My Team'] = my_team_name
        df['Opponent Team'] = opp_team_name
        df['My Score'] = df[my_score_col] if my_score_col in df.columns else 0
        df['Opp Score'] = df[opp_score_col] if opp_score_col in df.columns else 0
        df['My Scoremap'] = df[my_scoremap_col] if my_scoremap_col in df.columns else ""
        df['Opp Scoremap'] = df[opp_scoremap_col] if opp_scoremap_col in df.columns else ""
        df['Round Win'] = df[win_col] if win_col in df.columns else 0
        
        self.log(f"✅ {my_team_name} 관점 컬럼 생성 완료")
    
    def _aggregate_by_perspective(self, df, my_team_name):
        """관점 기반 집계"""
        
        group_cols = ['Match Name', 'Map', 'Round Number']
        
        agg_dict = {
            'Tournament': 'first',
            'Stage': 'first',
            'My Team': 'first',
            'Opponent Team': 'first',
            'Year': 'first',
            'My Score': 'first',
            'Opp Score': 'first',
            'My Scoremap': 'first',
            'Opp Scoremap': 'first',
            'Round Win': 'first',
        }
        
        # 경제 관련 (mean)
        for col in ['Loadout Value', 'Remaining Credits']:
            if col in df.columns:
                agg_dict[col] = 'mean'
        
        # 구매 전략 (first)
        if 'Buy Type Encoded' in df.columns:
            agg_dict['Buy Type Encoded'] = 'first'
        if 'Type' in df.columns:
            agg_dict['Type'] = 'first'
        
        # 킬/데스 관련 (sum)
        for col in df.columns:
            if 'kill' in col.lower() or 'death' in col.lower():
                agg_dict[col] = 'sum'
        
        df_aggregated = df.groupby(group_cols).agg(agg_dict).reset_index()
        
        self.log(f"   집계: {len(df_aggregated):,}행 × {len(df_aggregated.columns)}컬럼")
        
        return df_aggregated
    
    def run(self):
        """실행"""
        self.log("\n" + "╔" + "=" * 78 + "╗")
        self.log("║" + " " * 5 + "✅ Phase 2 개선: 양쪽 팀 관점 포함 (Dual Perspective)" + " " * 15 + "║")
        self.log("╚" + "=" * 78 + "╝")
        
        return self.aggregate_rounds()


# ============================================================
# 메인 실행
# ============================================================
if __name__ == "__main__":
    print("=" * 80)
    print("🔥 Phase 2 개선: 양쪽 팀 관점 모두 포함")
    print("   - Team A 관점: My Team = A, Opponent = B")
    print("   - Team B 관점: My Team = B, Opponent = A")
    print("=" * 80)
    
    aggregator = Phase2DualPerspectiveAggregation(
        merged_file='./data/final/all_matches_deduped.csv',
        output_dir='./data/final'
    )
    success = aggregator.run()
    
    if success:
        print("\n" + "=" * 80)
        print("✅ Phase 2 개선 완료!")
        print("=" * 80)
    else:
        print("\n" + "=" * 80)
        print("❌ Phase 2 개선 실패!")
        print("=" * 80)


🔥 Phase 2 개선: 양쪽 팀 관점 모두 포함
   - Team A 관점: My Team = A, Opponent = B
   - Team B 관점: My Team = B, Opponent = A
[2025-12-09 15:10:28] 
╔==============================================================================╗
[2025-12-09 15:10:28] ║     ✅ Phase 2 개선: 양쪽 팀 관점 포함 (Dual Perspective)               ║
[2025-12-09 15:10:28] ╚==============================================================================╝
[2025-12-09 15:10:28] ================================================================================
[2025-12-09 15:10:28] 📊 라운드 집계 (양쪽 팀 관점 포함)
[2025-12-09 15:10:28] ================================================================================


C:\Users\qkrgm\AppData\Local\Temp\ipykernel_6768\1747567629.py:70: DtypeWarning: Columns (20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.merged_file)


[2025-12-09 15:11:03] 
✅ 데이터 로드: 3,509,347행 × 26컬럼
[2025-12-09 15:11:03] 
🔍 사용 가능한 컬럼:

모든 컬럼:
   1. Tournament
   2. Stage
   3. Match Type
   4. Match Name
   5. Team A_score
   6. Team B_score
   7. Team A Score_score
   8. Team B Score_score
   9. Match Result
  10. Map
  11. Team A_map
  12. Duration
  13. Round Number
  14. Eliminator Team
  15. Eliminator
  16. Eliminator Agent
  17. Eliminated Team
  18. Eliminated
  19. Eliminated Agent
  20. Kill Type
  21. Team
  22. Loadout Value
  23. Remaining Credits
  24. Type
  25. Outcome
  26. Year
[2025-12-09 15:11:04] ✅ Round Number: 변환 완료
[2025-12-09 15:11:11] ✅ Loadout Value: 정제 완료
[2025-12-09 15:11:16] ✅ Remaining Credits: 정제 완료
[2025-12-09 15:11:19] ✅ Buy Type: 인코딩 완료
[2025-12-09 15:11:19] ✅ Round Outcome (Team B Win): 생성 완료
[2025-12-09 15:11:19] 

[2025-12-09 15:11:19] 🔵 Team A 관점 집계 중...
[2025-12-09 15:11:19] ================================================================================
[2025-12-09 15:11:21] ✅ Team A 관점 컬럼 

In [53]:

import pandas as pd
import os
from datetime import datetime
import re


class KillTypeParserFixed:
    """Kill Type에서 킬수 추출 (정규표현식)"""
    
    def __init__(self, input_file='./data/final/round_aggregated_full.csv',
                 output_dir='./data/final'):
        self.input_file = input_file
        self.output_dir = output_dir
        self.output_file = os.path.join(output_dir, 'round_aggregated_with_kills.csv')
        self.log_file = os.path.join(output_dir, 'kill_parsing_fixed.log')
        
        os.makedirs(output_dir, exist_ok=True)
        
        with open(self.log_file, 'w', encoding='utf-8') as f:
            f.write(f"Kill Type 파싱 (수정) 시작: {datetime.now()}\n")
    
    def log(self, msg):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_msg = f"[{timestamp}] {msg}"
        print(log_msg)
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(log_msg + "\n")
    
    def parse_kills(self, kill_type_str):
        """Kill Type 문자열에서 킬수 추출 (정규표현식)"""
        if pd.isna(kill_type_str):
            return 0
        
        kill_type_str = str(kill_type_str).strip()
        
        if not kill_type_str or kill_type_str == '0':
            return 0
        
        # 🔥 정규표현식: "2k", "3k", "4k", "5k", "1v1", "1v2", "2v1" 등 패턴
        # 숫자 + (k 또는 v + 숫자) 패턴
        pattern = r'\d+(?:k|v\d)'
        
        matches = re.findall(pattern, kill_type_str, re.IGNORECASE)
        
        return len(matches)
    
    def process_kills(self):
        """Kill Type으로부터 킬수 계산"""
        self.log("=" * 80)
        self.log("📊 Kill Type 파싱 (정규표현식)")
        self.log("=" * 80)
        
        try:
            df = pd.read_csv(self.input_file)
            self.log(f"\n✅ 데이터 로드: {len(df):,}행")
            
            # Kill Type 확인
            if 'Kill Type' not in df.columns:
                self.log("❌ Kill Type 컬럼 없음!")
                return False
            
            self.log(f"\n🔍 Kill Type 샘플 (파싱 결과):")
            samples = df['Kill Type'].unique()[:15]
            for kill_type in samples:
                kills = self.parse_kills(kill_type)
                self.log(f"   '{kill_type}' → {kills}킬")
            
            # 킬수 계산
            self.log(f"\n🔧 킬수 계산 중...")
            df['My Kills'] = df['Kill Type'].apply(self.parse_kills)
            
            self.log(f"✅ 킬수 계산 완료:")
            self.log(f"   Min: {df['My Kills'].min()}")
            self.log(f"   Max: {df['My Kills'].max()}")
            self.log(f"   Mean: {df['My Kills'].mean():.2f}")
            self.log(f"   Total: {df['My Kills'].sum():,}")
            
            # 분포 확인
            self.log(f"\n📊 킬수 분포:")
            kill_dist = df['My Kills'].value_counts().sort_index()
            for kills, count in kill_dist.items():
                pct = count / len(df) * 100
                self.log(f"   {int(kills)}킬: {count:,}개 ({pct:.1f}%)")
            
            # 샘플 확인
            self.log(f"\n📋 샘플 데이터:")
            sample_cols = ['Match Name', 'Map', 'Round Number', 'Kill Type', 'My Kills']
            for col in sample_cols:
                if col not in df.columns and col != 'My Kills':
                    sample_cols.remove(col)
            
            if 'My Kills' in df.columns:
                self.log(f"\n{df[sample_cols].head(10).to_string(index=False)}")
            
            # 저장
            df.to_csv(self.output_file, index=False, encoding='utf-8')
            self.log(f"\n✅ 저장: {self.output_file}")
            self.log(f"   컬럼: {len(df.columns)}")
            self.log(f"   'My Kills' 컬럼 추가됨")
            
            return True
        
        except Exception as e:
            self.log(f"❌ 에러: {str(e)}")
            import traceback
            self.log(traceback.format_exc())
            return False
    
    def run(self):
        """실행"""
        self.log("\n" + "╔" + "=" * 78 + "╗")
        self.log("║" + " " * 15 + "🔥 Kill Type 파싱 (정규표현식)" + " " * 31 + "║")
        self.log("╚" + "=" * 78 + "╝")
        
        return self.process_kills()


# ============================================================
# 메인 실행
# ============================================================
if __name__ == "__main__":
    print("=" * 80)
    print("🔥 Kill Type 파싱 (정규표현식): 킬수 정확하게 계산")
    print("=" * 80)
    
    parser = KillTypeParserFixed()
    success = parser.run()
    
    if success:
        print("\n" + "=" * 80)
        print("✅ Kill Type 파싱 완료!")
        print("=" * 80)
        print("\n📊 생성된 파일: round_aggregated_with_kills.csv")
        print("   - 'My Kills' 컬럼 정확하게 추가됨")
    else:
        print("\n" + "=" * 80)
        print("❌ Kill Type 파싱 실패!")
        print("=" * 80)

🔥 Kill Type 파싱 (정규표현식): 킬수 정확하게 계산
[2025-12-07 07:18:26] 
╔==============================================================================╗
[2025-12-07 07:18:26] ║               🔥 Kill Type 파싱 (정규표현식)                               ║
[2025-12-07 07:18:26] ╚==============================================================================╝
[2025-12-07 07:18:26] ================================================================================
[2025-12-07 07:18:26] 📊 Kill Type 파싱 (정규표현식)
[2025-12-07 07:18:26] ================================================================================
[2025-12-07 07:18:26] 
✅ 데이터 로드: 375,943행
[2025-12-07 07:18:26] 
🔍 Kill Type 샘플 (파싱 결과):
[2025-12-07 07:18:26]    '2k2k2k2k2k2k2k2k' → 8킬
[2025-12-07 07:18:26]    '2k2k2k2k1v11v1' → 6킬
[2025-12-07 07:18:26]    '2k2k2k2k' → 4킬
[2025-12-07 07:18:26]    '2k2k' → 2킬
[2025-12-07 07:18:26]    '3k3k3k3k3k3k' → 6킬
[2025-12-07 07:18:26]    '1v11v12k2k2k2k2k2k2k2k' → 10킬
[2025-12-07 07:18:26]    '2k2k2k2k3k3k3k3k3k3k' → 

In [21]:
import numpy as np
import pandas as pd

# CSV 로드
X_enc = pd.read_csv('./data/features_dedup/X_encoder.csv').values
X_dec = pd.read_csv('./data/features_dedup/X_decoder.csv').values
# 3D로 reshape: (샘플수, 12라운드, 7특성)
X_encoder = X_enc.reshape(-1, 12, 7)
X_decoder = X_dec.reshape(-1, 12, 7)

# ✅ 타겟: 6:7로 슬라이싱 (차원 유지)
y = X_decoder[:, :, 6:7]  # shape: (N, 12, 1)

# 예측용 입력 (참고용인듯?)
x_pred = X_encoder[:, :, 6:7] # shape: (N, 12, 1)

# 데이터 분할
X_train, X_temp, y_train, y_temp = train_test_split(X_encoder, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)



In [14]:
from sklearn.model_selection import train_test_split

# 80% 학습, 10% 검증, 10% 테스트
X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoder, y, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (21968, 12, 7), Val: (2746, 12, 7), Test: (2746, 12, 7)


In [13]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
import pickle


class ValorantAutoregTrainer:
    """
    깨끗한 1-step Autoregressive LSTM:
    - 입력: 지금까지의 라운드 시퀀스 (1~t)
    - 출력: 다음 라운드(t+1)의 my_win (0/1)
    - 학습/예측 모두 같은 규칙
    """

    def __init__(self,
                 all_rounds_path='./data/features_all/X_all.csv',
                 output_dir='./data/models_autoreg',
                 max_rounds=24,
                 n_features=7):
        self.all_rounds_path = all_rounds_path
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        self.max_rounds = max_rounds   # 예: 24라운드까지 있다고 가정
        self.n_features = n_features   # 예: 7개 피처
        self.win_index = 6             # my_win 컬럼 인덱스

        self.scaler = StandardScaler()

    # ====================================================
    # 1. 데이터 로드 및 "1-step" 학습 샘플 생성
    # ====================================================
    def load_and_build_dataset(self):
        print("📂 Step 1: 전체 라운드 데이터 로드 & 1-step 샘플 생성")

        # 가정: X_all.csv shape = (N, max_rounds * n_features)
        raw = pd.read_csv(self.all_rounds_path).values
        X_all = raw.reshape(-1, self.max_rounds, self.n_features)  # (N, T, F)
        N, T, F = X_all.shape
        print(f"[DEBUG] X_all shape: {X_all.shape}")

        # 스케일링 (win 컬럼 제외하고 해도 되지만, 여기선 전체)
        self.scaler.fit(X_all.reshape(-1, F))
        X_scaled = self.scaler.transform(X_all.reshape(-1, F)).reshape(X_all.shape)

        # 1-step 시퀀스 샘플 생성
        # 각 경기 샘플 i, 각 t=0..T-2 에 대해:
        #   input: X_scaled[i, :t+1, :] (길이 t+1)
        #   target: my_win at t+1 (X_all[i, t+1, win_index])
        X_seqs = []
        y_labels = []
        seq_lengths = []

        for i in range(N):
            for t in range(T - 1):
                # t까지(0..t) 를 입력, t+1의 승패를 타겟으로
                x_seq = X_scaled[i, :t+1, :]           # (t+1, F)
                y_next = X_all[i, t+1, self.win_index] # 스케일 전 원본에서 승패 사용 (0/1)

                X_seqs.append(x_seq)
                y_labels.append(y_next)
                seq_lengths.append(t+1)

        X_seqs = np.array(X_seqs, dtype=object)  # 각 샘플 길이가 다르므로 object로 보관
        y_labels = np.array(y_labels).astype(np.float32)
        seq_lengths = np.array(seq_lengths, dtype=np.int32)

        print(f"[DEBUG] 생성된 1-step 샘플 수: {len(X_seqs)}")
        print(f"[DEBUG] 시퀀스 길이 범위: {seq_lengths.min()} ~ {seq_lengths.max()}")

        # 길이가 다른 시퀀스를 LSTM에 넣으려면 pad가 필요.
        # 여기서는 max_len = max_rounds 로 맞춰서 앞을 0 패딩.
        max_len = self.max_rounds
        X_padded = np.zeros((len(X_seqs), max_len, F), dtype=np.float32)

        for idx, (seq, L) in enumerate(zip(X_seqs, seq_lengths)):
            X_padded[idx, max_len - L:, :] = seq  # 뒤에 정렬(right padding), 앞을 0으로

        print(f"[DEBUG] 패딩 후 X shape: {X_padded.shape}")  # (num_samples, max_len, F)

        # Train/Val/Test split
        X_train, X_temp, y_train, y_temp = train_test_split(
            X_padded, y_labels, test_size=0.2, random_state=42, stratify=y_labels
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
        )

        print("\n[DEBUG] 데이터 분할 결과")
        print(f"  Train: {X_train.shape[0]} 샘플")
        print(f"  Val  : {X_val.shape[0]} 샘플")
        print(f"  Test : {X_test.shape[0]} 샘플")
        print(f"  y_train win 비율: {y_train.mean():.4f}")
        print(f"  y_val   win 비율: {y_val.mean():.4f}")
        print(f"  y_test  win 비율: {y_test.mean():.4f}")

        return X_train, y_train, X_val, y_val, X_test, y_test

    # ====================================================
    # 2. 모델 구축 (1-step binary classifier)
    # ====================================================
    def build_model(self):
        print("\n🏗️ Step 2: 1-step Autoregressive LSTM 모델 구축")

        inp = Input(shape=(self.max_rounds, self.n_features), name='round_seq_input')

        x = LSTM(128, return_sequences=True, name='lstm1')(inp)
        x = BatchNormalization(name='bn1')(x)
        x = Dropout(0.3, name='drop1')(x)

        # 마지막 타임스텝만 사용
        x = LSTM(64, return_sequences=False, name='lstm2')(x)
        x = BatchNormalization(name='bn2')(x)
        x = Dropout(0.3, name='drop2')(x)

        out = Dense(1, activation='sigmoid', name='next_win')(x)

        model = Model(inp, out, name='Autoregressive1Step')
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        model.summary()
        return model

    # ====================================================
    # 3. 학습
    # ====================================================
    def train(self, model, X_train, y_train, X_val, y_val):
        print("\n🚀 Step 3: 모델 학습")

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ModelCheckpoint(
                filepath=f'{self.output_dir}/best_autoreg.keras',
                monitor='val_loss',
                save_best_only=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-6
            )
        ]

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=64,
            callbacks=callbacks,
            verbose=1
        )

        self._plot_history(history)

        # Scaler 저장
        with open(f'{self.output_dir}/scaler.pkl', 'wb') as f:
            pickle.dump(self.scaler, f)
        print("✅ Scaler 저장 완료: scaler.pkl")

        return history

    # ====================================================
    # 4. 평가
    # ====================================================
    def evaluate(self, model, X_test, y_test):
        print("\n" + "="*80)
        print("📊 Step 4: 1-step Autoregressive 최종 평가")
        print("="*80 + "\n")

        y_prob = model.predict(X_test, verbose=1)
        y_pred = (y_prob > 0.5).astype(int)

        print("\n[DEBUG] 예측/정답 분포")
        print(f"  y_test mean (win 비율): {y_test.mean():.4f}")
        print(f"  y_prob mean (확률 평균): {y_prob.mean():.4f}")
        print(f"  y_pred win 비율        : {y_pred.mean():.4f}")

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        print(f"\n📈 최종 성능 요약 (1-step):")
        print(f"   - Accuracy : {acc:.2%}")
        print(f"   - Precision: {prec:.4f}")
        print(f"   - Recall   : {rec:.4f}")
        print(f"   - F1-Score : {f1:.4f}\n")

    # ====================================================
    # 5. 서비스용: 지금까지 라운드로 다음 라운드 예측
    # ====================================================
    def predict_next_round(self, model, past_rounds_array):
        """
        past_rounds_array: shape (L, n_features)
        - L: 지금까지 실제로 진행된 라운드 수 (1~max_rounds-1)
        - 이걸 0패딩해서 (1, max_rounds, n_features)로 바꿔서 넣음
        """
        L = past_rounds_array.shape[0]
        assert L <= self.max_rounds, "입력 길이가 max_rounds를 넘었습니다."

        # 스케일링 (학습할 때와 같은 scaler 사용)
        past_scaled = self.scaler.transform(past_rounds_array)

        X_input = np.zeros((1, self.max_rounds, self.n_features), dtype=np.float32)
        X_input[0, self.max_rounds - L:, :] = past_scaled  # 뒤에 정렬

        prob = model.predict(X_input, verbose=0)[0, 0]
        return prob  # 다음 라운드 win 확률 (0~1)

    # ====================================================
    # 6. 그래프
    # ====================================================
    def _plot_history(self, history):
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 2, 1)
        plt.plot(history.history['loss'], label='Train Loss')
        plt.plot(history.history['val_loss'], label='Val Loss')
        plt.legend()
        plt.title('Loss')

        plt.subplot(1, 2, 2)
        plt.plot(history.history['accuracy'], label='Train Acc')
        plt.plot(history.history['val_accuracy'], label='Val Acc')
        plt.legend()
        plt.title('Accuracy')

        plt.tight_layout()
        plt.savefig(f'{self.output_dir}/training_history.png')
        plt.close()
        print("✅ 학습 곡선 저장 완료: training_history.png")

    # ====================================================
    # 7. 전체 파이프라인 실행
    # ====================================================
    def run(self):
        X_train, y_train, X_val, y_val, X_test, y_test = self.load_and_build_dataset()
        model = self.build_model()
        self.train(model, X_train, y_train, X_val, y_val)

        best_model = tf.keras.models.load_model(f'{self.output_dir}/best_autoreg.keras')
        self.evaluate(best_model, X_test, y_test)

        print("\n" + "="*80)
        print("🎉 1-step Autoregressive 전체 파이프라인 완료!")
        print(f"   - 모델 및 결과 저장 위치: {self.output_dir}")
        print("="*80)


if __name__ == "__main__":
    trainer = ValorantAutoregTrainer()
    trainer.run()


📂 Step 1: 전체 라운드 데이터 로드 & 1-step 샘플 생성
[DEBUG] X_all shape: (27460, 24, 7)
[DEBUG] 생성된 1-step 샘플 수: 631580
[DEBUG] 시퀀스 길이 범위: 1 ~ 23
[DEBUG] 패딩 후 X shape: (631580, 24, 7)

[DEBUG] 데이터 분할 결과
  Train: 505264 샘플
  Val  : 63158 샘플
  Test : 63158 샘플
  y_train win 비율: 0.4095
  y_val   win 비율: 0.4095
  y_test  win 비율: 0.4094

🏗️ Step 2: 1-step Autoregressive LSTM 모델 구축
Model: "Autoregressive1Step"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 round_seq_input (InputLayer  [(None, 24, 7)]          0         
 )                                                               
                                                                 
 lstm1 (LSTM)                (None, 24, 128)           69632     
                                                                 
 bn1 (BatchNormalization)    (None, 24, 128)           512       
                                                                 
 drop1 (Drop

In [32]:
import pandas as pd
import os

enc_path = './data/features_dedup/X_encoder.csv'
dec_path = './data/features_dedup/X_decoder.csv'
out_dir = './data/features_all'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'X_all.csv')

X_enc = pd.read_csv(enc_path)
X_dec = pd.read_csv(dec_path)

# 같은 경기 순서/행 기준으로 전반/후반이 맞게 정렬되어 있다는 가정
X_all = pd.concat([X_enc, X_dec], axis=1)  # 열 방향으로 이어붙이기

print("X_enc shape:", X_enc.shape)
print("X_dec shape:", X_dec.shape)
print("X_all shape:", X_all.shape)

X_all.to_csv(out_path, index=False)
print("✅ 저장 완료:", out_path)


X_enc shape: (27460, 84)
X_dec shape: (27460, 84)
X_all shape: (27460, 168)
✅ 저장 완료: ./data/features_all\X_all.csv


In [7]:
import tensorflow as tf

print("GPUs:", tf.config.list_physical_devices('GPU'))
print("Built with CUDA:", tf.test.is_built_with_cuda())


GPUs: []
Built with CUDA: False


In [16]:
# export_model_once.py
import tensorflow as tf
import pickle
import os

# 1) 옛날에 저장해둔 모델/스케일러 경로
OLD_MODEL_PATH = "./data/models_autoreg/best_autoreg.keras"
OLD_SCALER_PATH = "./data/models_autoreg/scaler.pkl"

# 2) 새로 저장할 경로 (.h5 + 그대로 쓸 scaler)
NEW_MODEL_PATH = "./data/models_autoreg/best_autoreg_compat.h5"
NEW_SCALER_PATH = "./data/models_autoreg/scaler_compat.pkl"


def main():
    print("[export] loading old model:", OLD_MODEL_PATH)
    model = tf.keras.models.load_model(OLD_MODEL_PATH)
    print("[export] old model loaded:", type(model))

    os.makedirs(os.path.dirname(NEW_MODEL_PATH), exist_ok=True)

    # .h5 포맷으로 저장 (Keras 3에서도 읽을 수 있음)
    print("[export] saving new model (.h5):", NEW_MODEL_PATH)
    model.save(NEW_MODEL_PATH)
    print("[export] saved new model")

    # 스케일러도 이름만 바꿔서 복사
    print("[export] copying scaler:", OLD_SCALER_PATH)
    with open(OLD_SCALER_PATH, "rb") as f:
        scaler = pickle.load(f)

    with open(NEW_SCALER_PATH, "wb") as f:
        pickle.dump(scaler, f)

    print("[export] done:", NEW_MODEL_PATH, NEW_SCALER_PATH)


if __name__ == "__main__":
    main()


[export] loading old model: ./data/models_autoreg/best_autoreg.keras
[export] old model loaded: <class 'keras.engine.functional.Functional'>
[export] saving new model (.h5): ./data/models_autoreg/best_autoreg_compat.h5
[export] saved new model
[export] copying scaler: ./data/models_autoreg/scaler.pkl
[export] done: ./data/models_autoreg/best_autoreg_compat.h5 ./data/models_autoreg/scaler_compat.pkl


In [20]:
import subprocess, sys, os

output_file = "requirements_from_notebook.txt"

result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
    check=True,
)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(result.stdout)

print(f"✅ 현재 Jupyter 환경의 패키지 목록을 '{output_file}'로 저장했습니다.")


✅ 현재 Jupyter 환경의 패키지 목록을 'requirements_from_notebook.txt'로 저장했습니다.
